# TOFU 3B IDK+AP LoRA diagnostic

One-seed exploratory method diagnostic on the locked TOFU development profile blocks. It is a resource-adapted LoRA implementation of the published IDK+AP objective, not an exact reproduction of the paper's full-parameter A100 experiment.

The run uses the existing fixed evaluation margins and profile-block bootstrap. It performs no parameter search. It trains the final profile blocks only if every development gate passes. The IDK marker and exact-string rates are descriptive string screens, not semantic refusal judgments.

Protocol SHA-256: `650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049`. Use GPU T4 x2 with Internet enabled. No Hugging Face secret is required for the public Open-Unlearning checkpoint.

In [ ]:
from pathlib import Path
Path('/kaggle/working/idk_ap_lora').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/tofu_official_eval.py
#!/usr/bin/env python3
"""Evaluate pinned TOFU checkpoints with the core Open-Unlearning metrics.

This is inference-only. Full checkpoints receive answer likelihood, greedy
generation/ROUGE-L, and truth-ratio evaluation on the canonical TOFU sets.
The paired retain90 oracle checkpoints are scored on forget truth ratios only
to compute the benchmark's forget-quality comparison.
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import os
import random
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from scipy.stats import ks_2samp
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
UPSTREAM_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
PROFILE_QA_COUNT = 20
BOOTSTRAP_SEED = 42
BOOTSTRAP_DRAWS = 5000
MAX_LENGTH = 512
SYSTEM_PROMPT = "You are a helpful assistant."
TEMPLATE_DATE = "10 Apr 2025"
MAX_NEW_TOKENS = 200
IGNORE_INDEX = -100
EPSILON = 1e-10

CHECKPOINTS = {
    "llama_3_2_3b_full": {
        "label": "Llama 3.2 3B full",
        "family": "llama_3_2_3b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    },
    "llama_3_2_3b_retain90": {
        "label": "Llama 3.2 3B retain90 oracle",
        "family": "llama_3_2_3b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "revision": "2c35819e9a731d096df9abd6efbcc361870c5333",
    },
    "llama_3_1_8b_full": {
        "label": "Llama 3.1 8B full",
        "family": "llama_3_1_8b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    },
    "llama_3_1_8b_retain90": {
        "label": "Llama 3.1 8B retain90 oracle",
        "family": "llama_3_1_8b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "revision": "f02956a9f25a18f38ad7808fbefb2f957347273e",
    },
}

DATA_FILES = (
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
)


def get_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-official-eval/1.0"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def load_data() -> tuple[dict[str, list[dict]], dict[str, str]]:
    rows_by_file = {}
    hashes = {}
    for filename in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
        raw = get_bytes(url)
        rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
        if any(not isinstance(row, dict) for row in rows):
            raise ValueError(f"{filename} contains a non-object JSONL row")
        if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
            raise ValueError(f"{filename} has a missing question/answer string")
        rows_by_file[filename] = rows
        hashes[filename] = hashlib.sha256(raw).hexdigest()
    return rows_by_file, hashes


def build_eval_sets(data: dict[str, list[dict]]) -> dict[str, list[dict]]:
    """Build canonical evaluation rows and attach source profile blocks."""
    retain_source = data["retain90.json"]
    retain_index = collections.defaultdict(list)
    retain_question_answers = collections.defaultdict(set)
    for i, row in enumerate(retain_source):
        retain_index[(row["question"], row["answer"])].append(i)
        retain_question_answers[row["question"]].add(row["answer"])
    collision_indices = {
        i
        for i, row in enumerate(retain_source)
        if len(retain_question_answers[row["question"]]) > 1
    }

    def basic(rows: list[dict], split: str, source_indices: list[int] | None = None) -> list[dict]:
        result = []
        for i, row in enumerate(rows):
            source_i = i if source_indices is None else source_indices[i]
            result.append({
                "split": split,
                "eval_row_index": i,
                "source_row_index": source_i,
                "profile_block": source_i // PROFILE_QA_COUNT if split in {"forget10", "retain90", "holdout10"} else None,
                "question": row["question"],
                "answer": row["answer"],
                "ambiguous_question_collision": split == "retain90" and source_i in collision_indices,
                "paraphrased_answer": row.get("paraphrased_answer"),
                "perturbed_answers": row.get("perturbed_answer"),
            })
        return result

    forget = data["forget10_perturbed.json"]
    forget_source = data["forget10.json"]
    if len(forget) != len(forget_source) or any(
        (a["question"], a["answer"]) != (b["question"], b["answer"])
        for a, b in zip(forget_source, forget)
    ):
        raise ValueError("forget10_perturbed no longer matches forget10 in source order")

    retain = data["retain_perturbed.json"]
    retain_source_indices = []
    for row in retain:
        matches = retain_index.get((row["question"], row["answer"]), [])
        if len(matches) != 1:
            raise ValueError("A canonical retain_perturbed row does not map uniquely to retain90")
        retain_source_indices.append(matches[0])
    if sorted(retain_source_indices) != list(range(20 * PROFILE_QA_COUNT)):
        raise ValueError("Canonical retain_perturbed is not the pinned first-20-profile retain sample")

    return {
        "forget10": basic(forget, "forget10"),
        "retain90": basic(retain, "retain90", retain_source_indices),
        "holdout10": basic(data["holdout10.json"], "holdout10"),
        "real_authors": basic(data["real_authors_perturbed.json"], "real_authors"),
        "world_facts": basic(data["world_facts_perturbed.json"], "world_facts"),
    }


def ids(value) -> list[int]:
    if hasattr(value, "keys") and "input_ids" in value:
        value = value["input_ids"]
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(x) for x in value]


def encode_answer(tokenizer, question: str, answer: str) -> tuple[list[int], list[int]]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    kwargs = {"date_string": TEMPLATE_DATE}
    full_ids = ids(tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=False, **kwargs
    ))
    prompt_ids = ids(tokenizer.apply_chat_template(
        conversation[:-1], tokenize=True, add_generation_prompt=True, **kwargs
    ))
    if not full_ids or len(prompt_ids) > len(full_ids):
        raise ValueError("Invalid chat-template tokenization")
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokenization is not an exact prefix of the full conversation")
    if tokenizer.eos_token_id is not None and full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(int(tokenizer.eos_token_id))
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    if len(labels) != len(full_ids) or not any(x != IGNORE_INDEX for x in labels[1:]):
        raise ValueError("Assistant answer mask is empty or misaligned")
    return full_ids, labels


def encode_prompt(tokenizer, question: str) -> list[int]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    result = ids(tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        date_string=TEMPLATE_DATE,
    ))
    if not result:
        raise ValueError("Chat-template prompt was empty")
    return result


def collate_scoring(rows: list[tuple[list[int], list[int]]], pad_id: int):
    width = max(len(x[0]) for x in rows)
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    labels = torch.full((len(rows), width), IGNORE_INDEX, dtype=torch.long)
    for i, (tokens, targets) in enumerate(rows):
        n = len(tokens)
        input_ids[i, :n] = torch.tensor(tokens)
        attention[i, :n] = 1
        labels[i, :n] = torch.tensor(targets)
    return input_ids, attention, labels


def score_answers(model, encoded: list[tuple[list[int], list[int]]], pad_id: int, batch_size: int) -> list[dict]:
    device = model.get_input_embeddings().weight.device
    output_scores = []
    offset = 0
    batches = 0
    while offset < len(encoded):
        size = min(batch_size, len(encoded) - offset)
        batch = encoded[offset:offset + size]
        input_ids, attention, labels = collate_scoring(batch, pad_id)
        try:
            with torch.inference_mode():
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    use_cache=False,
                )
                logits = out.logits[:, :-1, :].float()
                targets = labels[:, 1:].to(logits.device)
                losses = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="none",
                ).reshape(targets.shape)
                tokens_per_row = (targets != IGNORE_INDEX).sum(dim=1)
                if torch.any(tokens_per_row == 0):
                    raise ValueError("Evaluation example has no scored assistant tokens")
                mean_loss = losses.sum(dim=1) / tokens_per_row
                for i in range(size):
                    value = float(mean_loss[i].item())
                    output_scores.append({
                        "avg_answer_token_loss": value,
                        "answer_probability": math.exp(-value),
                        "scored_answer_tokens": int(tokens_per_row[i].item()),
                    })
            offset += size
            batches += 1
            if batches % 100 == 0 or offset == len(encoded):
                print(f"Scored {offset}/{len(encoded)} answer candidates", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during scoring; reducing batch size to {batch_size}", flush=True)
    return output_scores


def collate_prompts(rows: list[list[int]], pad_id: int):
    width = max(map(len, rows))
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    for i, row in enumerate(rows):
        input_ids[i, -len(row):] = torch.tensor(row)
        attention[i, -len(row):] = 1
    return input_ids, attention


def generate_answers(model, tokenizer, prompts: list[list[int]], batch_size: int) -> list[str]:
    device = model.get_input_embeddings().weight.device
    generations = []
    offset = 0
    batches = 0
    model.config.use_cache = True
    while offset < len(prompts):
        size = min(batch_size, len(prompts) - offset)
        current = prompts[offset:offset + size]
        input_ids, attention = collate_prompts(current, int(tokenizer.pad_token_id))
        try:
            with torch.inference_mode():
                output = model.generate(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = output[:, input_ids.shape[1]:]
            texts = tokenizer.batch_decode(
                new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            generations.extend(text.strip() for text in texts)
            offset += size
            batches += 1
            if batches % 25 == 0 or offset == len(prompts):
                print(f"Generated {offset}/{len(prompts)} answers", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during generation; reducing batch size to {batch_size}", flush=True)
    return generations


def build_jobs(eval_sets: dict[str, list[dict]], role: str):
    score_jobs = []
    generation_jobs = []
    for split, rows in eval_sets.items():
        for row in rows:
            key = (split, row["eval_row_index"])
            if role == "full" or split == "forget10":
                score_jobs.append({**row, "variant": "answer", "candidate_answer": row["answer"]})
                if row["paraphrased_answer"]:
                    score_jobs.append({**row, "variant": "paraphrased_answer", "candidate_answer": row["paraphrased_answer"]})
                for i, candidate in enumerate(row["perturbed_answers"] or []):
                    score_jobs.append({**row, "variant": f"perturbed_{i}", "candidate_answer": candidate})
            if role == "full":
                generation_jobs.append(row)
    return score_jobs, generation_jobs


def score_rows_by_key(jobs: list[dict], scores: list[dict]) -> dict[tuple[str, int], dict[str, dict]]:
    result = collections.defaultdict(dict)
    for job, score in zip(jobs, scores):
        key = (job["split"], job["eval_row_index"])
        if job["variant"] in result[key]:
            raise ValueError(f"Duplicate answer candidate score for {key}/{job['variant']}")
        result[key][job["variant"]] = score
    return result


def bootstrap_cluster_ci(rows: list[dict], field: str, seed: int = BOOTSTRAP_SEED) -> list[float] | None:
    groups = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            groups[int(row["profile_block"])].append(float(row[field]))
    if not groups:
        return None
    group_values = list(groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        sampled = [rng.choice(group_values) for _ in group_values]
        values = [value for group in sampled for value in group]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))], estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))]]


def mean_field(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def summarize_examples(examples: list[dict]) -> dict:
    summary = {"n_questions": len(examples), "n_profile_blocks": len({r["profile_block"] for r in examples if r.get("profile_block") is not None})}
    fields = (
        "answer_probability",
        "rougeL_recall",
        "truth_ratio",
        "truth_ratio_score",
        "normalized_option_probability",
    )
    for field in fields:
        value = mean_field(examples, field)
        if value is not None:
            summary[f"mean_{field}"] = value
            ci = bootstrap_cluster_ci(examples, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
    return summary


def make_metrics(
    rows: list[dict],
    split: str,
    score_map: dict[tuple[str, int], dict[str, dict]],
    generations: dict[tuple[str, int], str],
    tokenizer,
    rouge: rouge_scorer.RougeScorer,
) -> list[dict]:
    metrics = []
    for row in rows:
        key = (split, row["eval_row_index"])
        scores = score_map.get(key, {})
        if "answer" not in scores:
            raise ValueError(f"Missing gold-answer score for {key}")
        output = {
            "split": split,
            "eval_row_index": row["eval_row_index"],
            "source_row_index": row["source_row_index"],
            "profile_block": row["profile_block"],
            "question_sha256": hashlib.sha256(row["question"].encode("utf-8")).hexdigest(),
            "source_answer": row["answer"],
            "reference_answer": row["answer"],
            "ambiguous_question_collision": row["ambiguous_question_collision"],
            "answer_probability": scores["answer"]["answer_probability"],
            "answer_token_loss": scores["answer"]["avg_answer_token_loss"],
            "answer_scored_tokens": scores["answer"]["scored_answer_tokens"],
        }
        if "paraphrased_answer" in scores:
            para = scores["paraphrased_answer"]
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"] or []))]
            output["paraphrased_answer_probability"] = para["answer_probability"]
            output["paraphrased_answer_token_loss"] = para["avg_answer_token_loss"]
            output["perturbed_answer_probabilities"] = [x["answer_probability"] for x in wrong]
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            if wrong:
                # Match the pinned Open-Unlearning implementation: exponentiate
                # the mean per-candidate answer loss, i.e. the geometric mean of
                # perturbed-answer probabilities.
                geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
                ratio = geometric_wrong_probability / (para["answer_probability"] + EPSILON)
                output["truth_ratio"] = ratio
                output["truth_ratio_score"] = min(ratio, 1.0 / (ratio + EPSILON)) if split == "forget10" else max(0.0, 1.0 - ratio)
        elif row["perturbed_answers"]:
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"]))]
            wrong_probabilities = [x["answer_probability"] for x in wrong]
            geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
            ratio = geometric_wrong_probability / (scores["answer"]["answer_probability"] + EPSILON)
            output["perturbed_answer_probabilities"] = wrong_probabilities
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            output["truth_ratio"] = ratio
            output["truth_ratio_score"] = max(0.0, 1.0 - ratio)
            output["normalized_option_probability"] = scores["answer"]["answer_probability"] / (
                scores["answer"]["answer_probability"] + sum(wrong_probabilities) + EPSILON
            )
        if key in generations:
            text = generations[key]
            full_ids, _ = encode_answer(tokenizer, row["question"], row["answer"])
            prompt_ids = encode_prompt(tokenizer, row["question"])
            input_text = tokenizer.decode(
                prompt_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            full_text = tokenizer.decode(
                full_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            reference = full_text.replace(input_text, "").strip()
            output["generation"] = text
            output["reference_answer"] = reference
            output["rougeL_recall"] = rouge.score(reference, text)["rougeL"].recall
        metrics.append(output)
    return metrics


def model_utility(metrics_by_split: dict[str, list[dict]]) -> dict:
    components = {
        "retain_probability": mean_field(metrics_by_split["retain90"], "answer_probability"),
        "retain_rougeL_recall": mean_field(metrics_by_split["retain90"], "rougeL_recall"),
        "retain_truth_ratio_score": mean_field(metrics_by_split["retain90"], "truth_ratio_score"),
        "real_authors_normalized_option_probability": mean_field(metrics_by_split["real_authors"], "normalized_option_probability"),
        "real_authors_rougeL_recall": mean_field(metrics_by_split["real_authors"], "rougeL_recall"),
        "real_authors_truth_ratio_score": mean_field(metrics_by_split["real_authors"], "truth_ratio_score"),
        "world_facts_normalized_option_probability": mean_field(metrics_by_split["world_facts"], "normalized_option_probability"),
        "world_facts_rougeL_recall": mean_field(metrics_by_split["world_facts"], "rougeL_recall"),
        "world_facts_truth_ratio_score": mean_field(metrics_by_split["world_facts"], "truth_ratio_score"),
    }
    if any(value is None for value in components.values()):
        raise ValueError(f"Cannot compute complete TOFU model utility; missing component: {components}")
    if any(value <= 0 for value in components.values()):
        aggregate = 0.0
    else:
        aggregate = len(components) / sum(1.0 / value for value in components.values())
    return {"components": components, "harmonic_mean": aggregate}


def write_reports(output_dir: Path) -> None:
    keys = ("llama_3_2_3b_full", "llama_3_1_8b_full")
    results = {key: json.loads((output_dir / f"{key}_official_eval.json").read_text(encoding="utf-8")) for key in keys}
    summaries = {key: results[key]["summary"] for key in keys}
    split_labels = {
        "forget10": "Forget10",
        "retain90": "Canonical retain eval",
        "holdout10": "Holdout10",
        "real_authors": "Real authors",
        "world_facts": "World facts",
    }
    lines = [
        "# TOFU official-metric pre-unlearning baseline",
        "",
        "**Status:** complete and inference-only. It evaluates the published full checkpoints and paired retain90 oracle checkpoints. No NPO training or research adapter was run.",
        "",
        "The metric formulas, chat preprocessing, and generated-reference extraction follow the pinned Open-Unlearning implementation. `answer_probability` is mean per-answer-token probability; `rougeL_recall` scores greedy generated answers; Truth Ratio compares the geometric mean likelihood of perturbed answers with the paraphrased correct answer (or source correct answer for real-author/world-fact utility). Kaggle T4 uses FP16 + SDPA here, while upstream configs specify BF16 + FlashAttention 2; formulas and prompts match, but numerical generations are not bitwise-identical to the upstream runtime.",
        "",
        "## Full-checkpoint metrics",
        "",
        "| Split | Model | Questions / profile blocks | Mean answer probability (95% profile CI) | Mean ROUGE-L recall (95% profile CI) | Mean truth ratio | TOFU truth-ratio score | Normalized option probability |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for split, label in split_labels.items():
        for key in keys:
            summary = summaries[key][split]
            def render(metric: str, ci_key: str | None = None) -> str:
                val = summary.get(f"mean_{metric}")
                if val is None:
                    return "—"
                if ci_key and ci_key in summary:
                    low, high = summary[ci_key]
                    return f"{val:.4f} [{low:.4f}, {high:.4f}]"
                return f"{val:.4f}"
            n = f"{summary['n_questions']} / {summary['n_profile_blocks']}" if summary["n_profile_blocks"] else f"{summary['n_questions']} / —"
            lines.append(
                f"| {label} | {results[key]['model']['label']} | {n} | "
                f"{render('answer_probability', 'profile_cluster_bootstrap_95pct_ci_answer_probability')} | "
                f"{render('rougeL_recall', 'profile_cluster_bootstrap_95pct_ci_rougeL_recall')} | "
                f"{render('truth_ratio')} | {render('truth_ratio_score')} | {render('normalized_option_probability')} |"
            )
    lines += ["", "## Composite utility and forget-quality comparison", ""]
    lines.append("| Model | TOFU model-utility harmonic mean | Forget-quality KS p-value vs retain90 oracle |")
    lines.append("|---|---:|---:|")
    for key in keys:
        utility = results[key]["model_utility"]["harmonic_mean"]
        fq = results[key].get("forget_quality_vs_retain90_oracle", {}).get("p_value")
        lines.append(f"| {results[key]['model']['label']} | {utility:.6f} | {fq:.6g}" if fq is not None else f"| {results[key]['model']['label']} | {utility:.6f} | pending oracle result |")

    lines += [
        "",
        "## Retain-question collision sensitivity",
        "",
        "The pinned `retain90` data repeats one generic question with different answers. The canonical 400-row retain evaluation includes one of the two conflicting rows. Primary TOFU metrics retain the official row; the sensitivity row below excludes it.",
        "",
        "| Model | Retain probability, primary → excluding collision | Retain ROUGE-L recall, primary → excluding collision | Retain truth-ratio score, primary → excluding collision |",
        "|---|---:|---:|---:|",
    ]
    for key in keys:
        rows = results[key]["per_example"]["retain90"]
        clean = [row for row in rows if not row["ambiguous_question_collision"]]
        all_summary = summarize_examples(rows)
        clean_summary = summarize_examples(clean)
        lines.append(
            f"| {results[key]['model']['label']} | {all_summary['mean_answer_probability']:.4f} → {clean_summary['mean_answer_probability']:.4f} | "
            f"{all_summary['mean_rougeL_recall']:.4f} → {clean_summary['mean_rougeL_recall']:.4f} | "
            f"{all_summary['mean_truth_ratio_score']:.4f} → {clean_summary['mean_truth_ratio_score']:.4f} |"
        )
    lines += [
        "",
        "## Interpretation limits",
        "",
        "- Confidence intervals resample the 20 inferred profile blocks for forget10, canonical retain, and holdout10. The JSON contains question-level outputs; real-author and world-fact rows have no explicit cluster IDs, so they are reported without profile-cluster intervals.",
        "- Inference uses FP16 + SDPA because Kaggle T4 does not provide the upstream BF16 + FlashAttention 2 runtime. Small numeric and generation differences from an upstream evaluation are possible.",
        "- The 3B and 8B checkpoints come from different Llama generations and training histories. Their paired differences are descriptive checkpoint comparisons, not causal evidence about parameter count.",
        "- A small/large ROUGE or likelihood value alone does not prove forgetting. The forget-quality p-value compares distributions with a retain-only oracle and is not a standalone proof of equivalence or privacy.",
        "- TOFU profiles are fictional. These results cannot establish deletion of real personal data or real-world privacy protection.",
        "- The earlier 6 Nov likelihood-only run used a custom retain sample; this report uses the canonical `retain_perturbed` evaluation set from the pinned benchmark revision.",
        "",
        "## Reproduction metadata",
        "",
        f"Dataset revision: `{DATASET_REVISION}`. Open-Unlearning code revision: `{UPSTREAM_REVISION}`. Bootstrap: {BOOTSTRAP_DRAWS:,} profile-cluster draws, seed {BOOTSTRAP_SEED}. Greedy generation: max {MAX_NEW_TOKENS} new tokens.",
        "",
    ]
    (output_dir / "official_metric_baseline_report.md").write_text("\n".join(lines), encoding="utf-8")

    compare = {}
    for split in split_labels:
        left = results["llama_3_2_3b_full"]["per_example"][split]
        right = results["llama_3_1_8b_full"]["per_example"][split]
        right_by_index = {row["source_row_index"]: row for row in right}
        if len(right_by_index) != len(right) or {row["source_row_index"] for row in left} != set(right_by_index):
            raise ValueError(f"Full checkpoints do not have matched source examples in {split}")
        item = {}
        for metric in ("answer_probability", "rougeL_recall", "truth_ratio", "truth_ratio_score", "normalized_option_probability"):
            pairs = [(row, right_by_index[row["source_row_index"]]) for row in left if metric in row and metric in right_by_index[row["source_row_index"]]]
            if not pairs:
                continue
            deltas = [float(b[metric]) - float(a[metric]) for a, b in pairs]
            metric_result = {"n_questions": len(pairs), "mean_8b_minus_3b": sum(deltas) / len(deltas)}
            grouped = collections.defaultdict(list)
            for (a, _), delta in zip(pairs, deltas):
                if a.get("profile_block") is not None:
                    grouped[int(a["profile_block"])].append(delta)
            if grouped:
                group_ids = list(grouped)
                rng = random.Random(BOOTSTRAP_SEED)
                estimates = []
                for _ in range(BOOTSTRAP_DRAWS):
                    sampled = [rng.choice(group_ids) for _ in group_ids]
                    vals = [value for group in sampled for value in grouped[group]]
                    estimates.append(sum(vals) / len(vals))
                estimates.sort()
                metric_result["profile_cluster_bootstrap_95pct_ci"] = [
                    estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
                    estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
                ]
            else:
                metric_result["interval"] = None
                metric_result["interval_note"] = "No explicit profile cluster IDs; no inferential interval reported."
            item[metric] = metric_result
        compare[split] = item
    compare["interpretation_limit"] = "Descriptive paired checkpoint contrasts only; model generations/training histories differ."
    (output_dir / "paired_full_checkpoint_comparison.json").write_text(json.dumps(compare, indent=2) + "\n", encoding="utf-8")


def encode_jobs(tokenizer, jobs: list[dict]) -> list[tuple[list[int], list[int]]]:
    return [encode_answer(tokenizer, job["question"], job["candidate_answer"]) for job in jobs]


def evaluate_checkpoint(
    key: str,
    output_dir: Path,
    eval_sets: dict[str, list[dict]],
    dataset_hashes: dict[str, str],
    score_batch_size: int,
    generation_batch_size: int,
    max_memory_per_gpu_gib: float,
    adapter_path: Path | None = None,
    result_key: str | None = None,
    adapter_seed: int | None = None,
    profile_cohort_mode: bool = False,
) -> dict:
    from accelerate import __version__ as accelerate_version
    from transformers import __version__ as transformers_version

    spec = CHECKPOINTS[key]
    cache_root = Path(
        os.environ.get(
            "TOFU_MODEL_CACHE_ROOT",
            str(Path(tempfile.gettempdir()) / "tofu_official_eval_cache"),
        )
    )
    model_cache = cache_root / key
    model_cache.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], cache_dir=str(model_cache), use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if not tokenizer.is_fast:
        raise RuntimeError("Fast tokenizer required")

    score_jobs, generation_jobs = build_jobs(eval_sets, spec["role"])
    encoded_jobs = encode_jobs(tokenizer, score_jobs)
    prompt_ids = [encode_prompt(tokenizer, row["question"]) for row in generation_jobs]
    # Transformers 5.22's allocator warmup reserves a second buffer roughly
    # the size of the 8B model shard. On T4x2 this OOMs before weights load.
    # Disable only this load-time optimization for 8B; checkpoint weights still
    # load normally. Keep the library default for the smaller 3B model.
    allocator_warmup_strategy = "library default"
    if spec["family"] == "llama_3_1_8b":
        import transformers.modeling_utils as modeling_utils

        if not hasattr(modeling_utils, "caching_allocator_warmup"):
            raise RuntimeError("Pinned Transformers version lacks the expected allocator warmup hook")
        modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None
        allocator_warmup_strategy = "disabled for 8B to avoid duplicate preallocation; weights load normally"

    memory_limit = f"{max_memory_per_gpu_gib:g}GiB"
    # With only two T4s, balanced_low_0 put almost all 8B decoder blocks on
    # GPU 1 and caused CPU offload. Even balancing leaves ample generation
    # headroom on both GPUs at the configured 13 GiB cap.
    print(f"Loading {spec['label']} with balanced; max_memory={memory_limit} per T4", flush=True)
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="balanced",
        max_memory={0: memory_limit, 1: memory_limit, "cpu": "40GiB"},
        attn_implementation="sdpa",
    )
    device_map = getattr(model, "hf_device_map", {})
    offloaded_devices = sorted({str(device) for device in device_map.values() if str(device).lower() in {"cpu", "disk"}})
    print({"loaded_model": spec["label"], "hf_device_map": device_map, "offloaded_devices": offloaded_devices}, flush=True)
    if spec["family"] == "llama_3_1_8b" and offloaded_devices:
        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise RuntimeError(f"8B placement used CPU/disk offload {offloaded_devices}; refusing a slow, unbounded evaluation")
    result_spec = dict(spec)
    if adapter_path is not None:
        from peft import PeftModel

        if not (adapter_path / "adapter_config.json").is_file():
            raise FileNotFoundError(f"LoRA adapter config missing from {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        result_spec["label"] = f"{spec['label']} NPO LoRA seed {adapter_seed}"
        result_spec["adapter"] = {
            "method": "NPO + retain NLL, LoRA adapter",
            "seed": adapter_seed,
            "adapter_dir_name": adapter_path.name,
            "files_sha256": {
                p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in sorted(adapter_path.iterdir()) if p.is_file()
            },
        }
    model.eval()
    model.config.use_cache = False

    # A tiny non-research prompt tests generation before the full dataset pass.
    smoke_ids = encode_prompt(tokenizer, "Reply with one short greeting.")
    smoke_tensor = torch.tensor([smoke_ids], dtype=torch.long, device=model.get_input_embeddings().weight.device)
    smoke_started = time.monotonic()
    model.config.use_cache = True
    with torch.inference_mode():
        smoke_output = model.generate(
            input_ids=smoke_tensor,
            attention_mask=torch.ones_like(smoke_tensor),
            do_sample=False,
            max_new_tokens=4,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False
    smoke_new_tokens = int(smoke_output.shape[1] - smoke_tensor.shape[1])
    if smoke_new_tokens < 1:
        raise RuntimeError("GPU placement smoke generated no tokens; refusing full evaluation")
    placement_smoke = {
        "status": "PASS",
        "new_tokens": smoke_new_tokens,
        "elapsed_seconds": round(time.monotonic() - smoke_started, 2),
        "research_rows_used": 0,
    }
    print({"placement_smoke": placement_smoke}, flush=True)

    print(f"Starting answer scoring: {len(encoded_jobs)} candidates", flush=True)
    score_values = score_answers(model, encoded_jobs, int(tokenizer.pad_token_id), score_batch_size)
    if len(score_values) != len(score_jobs):
        raise RuntimeError("Scored row count does not equal job count")
    score_map = score_rows_by_key(score_jobs, score_values)
    generations = {}
    if spec["role"] == "full":
        print(f"Starting greedy generation: {len(prompt_ids)} prompts", flush=True)
        generated_texts = generate_answers(model, tokenizer, prompt_ids, generation_batch_size)
        if len(generated_texts) != len(generation_jobs):
            raise RuntimeError("Generation row count does not equal prompt count")
        generations = {
            (row["split"], row["eval_row_index"]): text
            for row, text in zip(generation_jobs, generated_texts)
        }

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge.tokenizer = tokenizer
    splits_to_write = ("forget10",) if spec["role"] == "retain90_oracle" else tuple(eval_sets)
    metrics_by_split = {
        split: make_metrics(eval_sets[split], split, score_map, generations, tokenizer, rouge)
        for split in splits_to_write
    }
    result = {
        "status": (
            "PASS_PROFILE_COHORT_EVAL" if profile_cohort_mode
            else "PASS_OFFICIAL_TOFU_ADAPTER_EVAL" if adapter_path is not None
            else "PASS_OFFICIAL_TOFU_BASELINE"
        ),
        "model": result_spec,
        **({"study_type": "exploratory profile-cohort follow-up; not a standard full-split TOFU evaluation"} if profile_cohort_mode else {}),
        "evaluation": {
            "dataset_id": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "dataset_sha256_by_file": dataset_hashes,
            "open_unlearning_revision": UPSTREAM_REVISION,
            "profile_block_size": PROFILE_QA_COUNT,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "bootstrap_draws": BOOTSTRAP_DRAWS,
            "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "use_cache": True},
            "declared_max_input_length": MAX_LENGTH,
            "input_truncation": False,
            "max_scored_sequence_tokens": max((len(tokens) for tokens, _ in encoded_jobs), default=0),
            "scored_sequences_over_declared_max_length": sum(len(tokens) > MAX_LENGTH for tokens, _ in encoded_jobs),
            "max_generation_prompt_tokens": max((len(tokens) for tokens in prompt_ids), default=0),
            **({
                "profile_cohort_mode": True,
                "rows_by_split": {split: len(rows) for split, rows in eval_sets.items()},
                "profile_blocks_by_split": {
                    split: sorted({row["profile_block"] for row in rows if row.get("profile_block") is not None})
                    for split, rows in eval_sets.items()
                },
                "metric_implementation": "Pinned TOFU answer probability, greedy-generation ROUGE-L recall, and forget Truth Ratio; profile-cohort subset, not a standard full-split benchmark evaluation.",
            } if profile_cohort_mode else {
                "canonical_retain_eval_rows": len(eval_sets["retain90"]),
                "canonical_retain_profile_blocks": sorted({row["profile_block"] for row in eval_sets["retain90"]}),
                "metric_implementation": "Pinned Open-Unlearning probability, ROUGE-L recall, Truth Ratio, normalized option probability, model-utility harmonic mean, and KS forget-quality definitions; see README for exact conventions.",
            }),
            "score_jobs": len(score_jobs),
            "generation_jobs": len(generation_jobs),
        },
        "summary": {split: summarize_examples(rows) for split, rows in metrics_by_split.items()},
        "per_example": {split: rows for split, rows in metrics_by_split.items()},
        "model_utility": model_utility(metrics_by_split) if spec["role"] == "full" and not profile_cohort_mode else None,
        "runtime": {
            "torch_version": torch.__version__,
            "transformers_version": transformers_version,
            "accelerate_version": accelerate_version,
            "rouge_score_version": __import__("importlib.metadata", fromlist=["version"]).version("rouge-score"),
            "scipy_version": __import__("scipy").__version__,
            "inference_dtype": "float16",
            "max_memory_per_gpu_gib": max_memory_per_gpu_gib,
            "device_map_strategy": "balanced",
            "hf_device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_offload_used": bool(offloaded_devices),
            "allocator_warmup_strategy": allocator_warmup_strategy,
            "placement_smoke": placement_smoke,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
            "elapsed_seconds": round(time.monotonic() - started, 2),
            "training_started": False,
            "research_adapter_created": adapter_path is not None,
        },
    }
    result_key = result_key or key
    suffix = "_profile_eval.json" if profile_cohort_mode else "_official_eval.json"
    result_path = output_dir / f"{result_key}{suffix}"
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    score_path = output_dir / f"{result_key}_examples.jsonl"
    with score_path.open("w", encoding="utf-8") as f:
        for rows in metrics_by_split.values():
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    del model, tokenizer, encoded_jobs, prompt_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if os.environ.get("TOFU_KEEP_MODEL_CACHE") != "1":
        shutil.rmtree(model_cache, ignore_errors=True)
    return result


def add_forget_quality(results_dir: Path, full_key: str, oracle_key: str) -> None:
    full_path = results_dir / f"{full_key}_official_eval.json"
    oracle_path = results_dir / f"{oracle_key}_official_eval.json"
    if not full_path.exists() or not oracle_path.exists():
        return
    full = json.loads(full_path.read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    full_rows = full["per_example"]["forget10"]
    oracle_rows = oracle["per_example"]["forget10"]
    if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [
        (r["source_row_index"], r["question_sha256"]) for r in oracle_rows
    ]:
        raise ValueError("Full and retain90 oracle forget rows are not aligned")
    stat = ks_2samp(
        [1.0 / (row["truth_ratio"] + EPSILON) for row in full_rows],
        [1.0 / (row["truth_ratio"] + EPSILON) for row in oracle_rows],
    )
    full["forget_quality_vs_retain90_oracle"] = {
        "metric": "KS p-value on inverse forget truth-ratio distributions; monotonic inversion leaves the two-sample KS statistic and p-value unchanged",
        "n_full_checkpoint": len(full_rows),
        "n_retain90_oracle": len(oracle_rows),
        "ks_statistic": float(stat.statistic),
        "p_value": float(stat.pvalue),
        "interpretation": "A larger p-value means these two empirical distributions are less distinguishable by this KS test; it is not proof of successful unlearning.",
    }
    full_path.write_text(json.dumps(full, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/tofu_official_eval"))
    parser.add_argument("--models", default=",".join(CHECKPOINTS))
    parser.add_argument("--score-batch-size", type=int, default=2)
    parser.add_argument("--generation-batch-size", type=int, default=2)
    parser.add_argument("--max-memory-per-gpu-gib", type=float, default=13.0)
    parser.add_argument("--resume-existing", action="store_true", help="Reuse passing, pinned result files already in output-dir; final verifier checks every row and metric.")
    args = parser.parse_args()
    selected = [x.strip() for x in args.models.split(",") if x.strip()]
    if not selected or set(selected) - set(CHECKPOINTS):
        parser.error(f"models must be selected from {sorted(CHECKPOINTS)}")
    if args.max_memory_per_gpu_gib <= 0 or args.max_memory_per_gpu_gib >= 14:
        parser.error("max-memory-per-gpu-gib must be >0 and <14 to leave T4 runtime headroom")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Requires GPU T4 x2; stopped before loading model weights")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    data, hashes = load_data()
    eval_sets = build_eval_sets(data)
    status_path = args.output_dir / "official_eval_status.json"
    reused_results = {}
    if args.resume_existing and status_path.exists():
        previous = json.loads(status_path.read_text(encoding="utf-8"))
        if previous.get("dataset_hashes") != hashes:
            raise RuntimeError("Cannot resume: existing receipt has different pinned dataset hashes")
        for key, entry in previous.get("model_results", {}).items():
            if entry.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
                continue
            result_path = args.output_dir / f"{key}_official_eval.json"
            examples_path = args.output_dir / f"{key}_examples.jsonl"
            if key not in CHECKPOINTS or not result_path.is_file() or not examples_path.is_file():
                raise RuntimeError(f"Cannot resume: incomplete successful receipt for {key}")
            prior_result = json.loads(result_path.read_text(encoding="utf-8"))
            spec = CHECKPOINTS[key]
            if (
                prior_result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE"
                or prior_result.get("model", {}).get("repo_id") != spec["repo_id"]
                or prior_result.get("model", {}).get("revision") != spec["revision"]
                or prior_result.get("model", {}).get("role") != spec["role"]
                or prior_result.get("evaluation", {}).get("dataset_sha256_by_file") != hashes
            ):
                raise RuntimeError(f"Cannot resume: pinned model/data metadata mismatch for {key}")
            reused_results[key] = entry
        if set(selected) & set(reused_results):
            raise RuntimeError("Requested model is already marked successful; remove it from --models or start a clean output directory")

    for key in selected:
        if CHECKPOINTS[key]["role"] == "full":
            oracle_key = key.replace("_full", "_retain90")
            if oracle_key not in selected and oracle_key not in reused_results:
                parser.error(f"{key} requires its matching retain90 oracle in --models or a verified resumed output")

    status = {
        "status": "RUNNING",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "selected_models": selected,
        "reused_models": sorted(reused_results),
        "dataset_hashes": hashes,
        "model_results": reused_results,
        "training_started": False,
    }
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    for key in selected:
        try:
            result = evaluate_checkpoint(
                key,
                args.output_dir,
                eval_sets,
                hashes,
                args.score_batch_size,
                args.generation_batch_size,
                args.max_memory_per_gpu_gib,
            )
            status["model_results"][key] = {
                "status": result["status"],
                "result_file": f"{key}_official_eval.json",
                "examples_file": f"{key}_examples.jsonl",
                "runtime_seconds": result["runtime"]["elapsed_seconds"],
            }
        except Exception as error:
            shutil.rmtree(Path(tempfile.gettempdir()) / "tofu_official_eval_cache" / key, ignore_errors=True)
            status["status"] = "ERROR"
            status["error"] = f"{type(error).__name__}: {error}"
            status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            raise
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"{key}: PASS_OFFICIAL_TOFU_BASELINE", flush=True)
    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in status["model_results"] and oracle_key in status["model_results"]:
            add_forget_quality(args.output_dir, full_key, oracle_key)
    complete_keys = set(CHECKPOINTS)
    if complete_keys <= set(status["model_results"]):
        write_reports(args.output_dir)
    else:
        status["status"] = "PARTIAL"
        status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"Partial results saved under {args.output_dir}; missing {sorted(complete_keys - set(status['model_results']))}")
        return
    status["status"] = "COMPLETE"
    status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Results saved under {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/evaluate_npo_profile_cohort.py
#!/usr/bin/env python3
"""Evaluate and gate the preregistered TOFU NPO-LoRA profile cohorts."""

from __future__ import annotations

import argparse
import hashlib
import json
import random
from pathlib import Path

from tofu_official_eval import (
    BOOTSTRAP_DRAWS,
    CHECKPOINTS,
    DATASET_REVISION,
    evaluate_checkpoint,
    load_data,
)


def expand(ranges: list[list[int]]) -> list[int]:
    return [block for first, last in ranges for block in range(first, last + 1)]


def make_rows(data: dict[str, list[dict]], split: str, profile_blocks: list[int]) -> list[dict]:
    profile_size = 20
    if split == "forget10":
        source = data["forget10.json"]
        eval_source = data["forget10_perturbed.json"]
        if len(source) != len(eval_source) or any(
            (a["question"], a["answer"]) != (b["question"], b["answer"])
            for a, b in zip(source, eval_source)
        ):
            raise ValueError("forget10_perturbed does not align with forget10 source order")
    elif split == "retain90":
        source = data["retain90.json"]
        eval_source = source
    elif split == "holdout10":
        source = data["holdout10.json"]
        eval_source = source
    else:
        raise ValueError(f"Unsupported profile split: {split}")

    selected_blocks = set(profile_blocks)
    output = []
    for source_index, row in enumerate(source):
        block = source_index // profile_size
        if block not in selected_blocks:
            continue
        paired = eval_source[source_index]
        output.append({
            "split": split,
            "eval_row_index": len(output),
            "source_row_index": source_index,
            "profile_block": block,
            "question": row["question"],
            "answer": row["answer"],
            "ambiguous_question_collision": False,
            "paraphrased_answer": paired.get("paraphrased_answer") if split == "forget10" else None,
            "perturbed_answers": paired.get("perturbed_answer") if split == "forget10" else [],
        })
    expected = len(selected_blocks) * profile_size
    if len(output) != expected:
        raise ValueError(f"{split}: expected {expected} selected rows, got {len(output)}")
    if split == "forget10" and any(not row["perturbed_answers"] for row in output):
        raise ValueError("A target forget row has no perturbed answers; Truth Ratio cannot be calculated")
    return output


def cohort_sets(data: dict[str, list[dict]], protocol: dict, cohort: str) -> dict[str, list[dict]]:
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    return {
        split: make_rows(data, split, expand(assignments[f"{split}_{cohort}_utility_eval"] if split != "forget10" else assignments[f"forget10_{cohort}_train_and_eval"]))
        for split in ("forget10", "retain90", "holdout10")
    }


def reference_sets(data: dict[str, list[dict]], protocol: dict) -> dict[str, list[dict]]:
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    target = list(range(20))
    retain = expand(assignments["retain90_dev_utility_eval"]) + expand(assignments["retain90_final_utility_eval"])
    holdout = list(range(20))
    return {
        "forget10": make_rows(data, "forget10", target),
        "retain90": make_rows(data, "retain90", retain),
        "holdout10": make_rows(data, "holdout10", holdout),
    }


def paired_cluster_delta(base_rows: list[dict], candidate_rows: list[dict], field: str, seed: int = 42) -> dict:
    base_by_index = {int(row["source_row_index"]): row for row in base_rows}
    groups: dict[int, list[float]] = {}
    for row in candidate_rows:
        base = base_by_index.get(int(row["source_row_index"]))
        if base is None or field not in row or field not in base:
            continue
        groups.setdefault(int(row["profile_block"]), []).append(float(row[field]) - float(base[field]))
    if not groups:
        raise ValueError(f"No matched baseline/candidate rows for {field}")
    block_ids = sorted(groups)
    point = sum(sum(values) for values in groups.values()) / sum(len(values) for values in groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        selected = [rng.choice(block_ids) for _ in block_ids]
        values = [value for block in selected for value in groups[block]]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return {
        "n_questions": sum(len(values) for values in groups.values()),
        "n_profile_blocks": len(groups),
        "mean_delta_candidate_minus_full": point,
        "profile_cluster_bootstrap_95pct_ci": [
            estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
            estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
        ],
        "bootstrap_draws": BOOTSTRAP_DRAWS,
        "bootstrap_seed": seed,
    }


def gate_comparison(base: dict, candidate: dict, protocol: dict) -> dict:
    tests = {
        "forget10.truth_ratio_score": ("forget10", "truth_ratio_score", "lower_gt_zero"),
        "forget10.answer_probability": ("forget10", "answer_probability", "upper_lt_zero"),
        "retain90.answer_probability": ("retain90", "answer_probability", "lower_gt_minus_margin"),
        "retain90.rougeL_recall": ("retain90", "rougeL_recall", "lower_gt_minus_margin"),
        "holdout10.answer_probability": ("holdout10", "answer_probability", "lower_gt_minus_margin"),
    }
    margin = float(protocol["selection"]["noninferiority_margin"])
    checks = {}
    for name, (split, field, rule) in tests.items():
        delta = paired_cluster_delta(base["per_example"][split], candidate["per_example"][split], field)
        low, high = delta["profile_cluster_bootstrap_95pct_ci"]
        passed = low > 0 if rule == "lower_gt_zero" else high < 0 if rule == "upper_lt_zero" else low > -margin
        checks[name] = {
            "delta": delta["mean_delta_candidate_minus_full"],
            "ci95": [low, high],
            "rule": rule,
            "threshold": 0.0 if rule.endswith("zero") else -margin,
            "pass": bool(passed),
            "n_questions": delta["n_questions"],
            "n_profile_blocks": delta["n_profile_blocks"],
        }
    result = {
        "status": "PASS_DIAGNOSTIC_GATES" if all(item["pass"] for item in checks.values()) else "FAIL_DIAGNOSTIC_GATES",
        "interpretation": "One-seed exploratory profile-cohort follow-up; not confirmatory and not a standard full-split TOFU run.",
        "utility_and_forgetting_gates": checks,
        "all_gates_pass": all(item["pass"] for item in checks.values()),
        "noninferiority_margin": margin,
        "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": 42, "unit": "profile block, paired by source row index"},
    }
    return result


def load_adapter_receipt(adapter_dir: Path, cohort: str, alpha: float) -> dict:
    receipt_path = adapter_dir.parent / "training_receipt.json"
    if not receipt_path.is_file():
        raise FileNotFoundError(f"Adapter training receipt missing: {receipt_path}")
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    if receipt.get("status") != "TRAINING_COMPLETE" or not receipt.get("adapter_saved"):
        raise ValueError("Adapter receipt is not a complete training run")
    followup = receipt.get("followup", {})
    if followup.get("cohort") != cohort or float(followup.get("selected_alpha_retain", -1)) != alpha:
        raise ValueError("Adapter cohort/alpha does not match evaluation request")
    actual_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(adapter_dir.iterdir()) if path.is_file()
    }
    if actual_hashes != receipt.get("adapter_files_sha256"):
        raise ValueError("Adapter files do not match their training receipt hashes")
    return receipt


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--protocol", type=Path, default=Path(__file__).with_name("npo_lora_recovery_protocol.json"))
    parser.add_argument("--mode", choices=("reference", "dev", "final"), required=True)
    parser.add_argument("--alpha-retain", type=float)
    parser.add_argument("--adapter-dir", type=Path)
    parser.add_argument("--reference-eval", type=Path)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("protocol_id") not in {
        "tofu_3b_npo_lora_profile_recovery_v1",
        "tofu_3b_npo_lora_profile_alpha40_followup_v1",
    }:
        raise ValueError("Wrong profile recovery protocol")
    if protocol["status"] != "PREREGISTERED_BEFORE_GPU_RUN":
        raise ValueError("Recovery protocol is not preregistered")
    if (args.mode == "reference") != (args.adapter_dir is None):
        raise ValueError("Reference mode requires no adapter; dev/final modes require --adapter-dir")
    if args.mode != "reference" and args.alpha_retain not in protocol["selection"]["candidate_retain_nll_weights_ascending"]:
        raise ValueError("Adapter alpha is not in the locked candidate grid")

    data, hashes = load_data()
    for filename, expected_hash in protocol["data"]["sha256"].items():
        if hashes.get(filename) != expected_hash:
            raise ValueError(f"Pinned source hash mismatch: {filename}")
    evaluation_sets = reference_sets(data, protocol) if args.mode == "reference" else cohort_sets(data, protocol, args.mode)
    if args.mode == "reference":
        adapter_path = None
        result_key = "full_reference"
    else:
        load_adapter_receipt(args.adapter_dir, args.mode, float(args.alpha_retain))
        adapter_path = args.adapter_dir
        result_key = f"{args.mode}_alpha_{args.alpha_retain:g}"
    args.output_dir.mkdir(parents=True, exist_ok=True)
    result = evaluate_checkpoint(
        "llama_3_2_3b_full",
        args.output_dir,
        evaluation_sets,
        hashes,
        score_batch_size=1,
        generation_batch_size=1,
        max_memory_per_gpu_gib=12.5,
        adapter_path=adapter_path,
        result_key=result_key,
        adapter_seed=42 if adapter_path else None,
        profile_cohort_mode=True,
    )
    if result.get("status") != "PASS_PROFILE_COHORT_EVAL":
        raise RuntimeError("Profile cohort evaluation did not complete")
    if args.mode == "reference":
        return

    reference_path = args.reference_eval or (args.output_dir.parent / "reference_eval" / "full_reference_profile_eval.json")
    if not reference_path.is_file():
        raise FileNotFoundError(f"Full-checkpoint profile reference evaluation must be completed first: {reference_path}")
    baseline = json.loads(reference_path.read_text(encoding="utf-8"))
    comparison = gate_comparison(baseline, result, protocol)
    comparison.update({
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "model_revision": CHECKPOINTS["llama_3_2_3b_full"]["revision"],
        "dataset_revision": DATASET_REVISION,
        "cohort": args.mode,
        "alpha_retain": float(args.alpha_retain),
        "training_receipt_sha256": hashlib.sha256((args.adapter_dir.parent / "training_receipt.json").read_bytes()).hexdigest(),
        "evaluation_receipt": f"{result_key}_profile_eval.json",
    })
    comparison_path = args.output_dir / f"{args.mode}_alpha_{args.alpha_retain:g}_comparison.json"
    comparison_path.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(comparison, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/validate_protocol.py
#!/usr/bin/env python3
"""Fail-closed validator for the locked TOFU IDK+AP profile diagnostic."""

from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from urllib.request import Request, urlopen


EXPECTED_PROTOCOL_SHA256 = "650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049"

def sha256(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()


def fetch(filename: str, revision: str) -> tuple[bytes, list[dict]]:
    url = f"https://huggingface.co/datasets/locuslab/TOFU/resolve/{revision}/{filename}"
    request = Request(url, headers={"User-Agent": "fsds-idk-ap-protocol-audit/1.0"})
    with urlopen(request, timeout=60) as response:
        raw = response.read()
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    if any(not isinstance(row, dict) for row in rows):
        raise ValueError(f"{filename} contains a non-object row")
    if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
        raise ValueError(f"{filename} has invalid question/answer fields")
    return raw, rows


def expand(ranges: list[list[int]]) -> list[int]:
    blocks = [block for first, last in ranges for block in range(first, last + 1)]
    if len(blocks) != len(set(blocks)):
        raise ValueError("A profile assignment contains duplicate blocks")
    return blocks


def validate(protocol_path: Path, idk_path: Path) -> dict:
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise ValueError("Protocol is not marked preregistered before GPU work")
    if protocol.get("protocol_id") != "tofu_3b_idk_ap_lora_profile_diagnostic_v1":
        raise ValueError("Unexpected protocol ID")
    if sha256(protocol_path.read_bytes()) != EXPECTED_PROTOCOL_SHA256:
        raise ValueError("Locked protocol SHA-256 mismatch; create a new protocol version for any design changes")
    data = protocol["data"]
    if data["repo_id"] != "locuslab/TOFU" or data["revision"] != "324592d84ae4f482ac7249b9285c2ecdb53e3a68":
        raise ValueError("TOFU source does not match the locked revision")
    observed_hashes: dict[str, str] = {}
    rows_by_file: dict[str, list[dict]] = {}
    for filename, expected_count in data["expected_rows"].items():
        raw, rows = fetch(filename, data["revision"])
        observed_hashes[filename] = sha256(raw)
        if observed_hashes[filename] != data["sha256"][filename]:
            raise ValueError(f"TOFU file hash mismatch: {filename}")
        if len(rows) != int(expected_count):
            raise ValueError(f"TOFU row count mismatch: {filename}: {len(rows)}")
        if len(rows) % int(data["profile_block_size"]):
            raise ValueError(f"{filename} does not contain complete profile blocks")
        rows_by_file[filename] = rows

    assignments = data["profile_assignment_ranges_inclusive"]
    forget_dev = expand(assignments["forget10_dev_train_and_eval"])
    forget_final = expand(assignments["forget10_final_train_and_eval"])
    retain_train = expand(assignments["retain90_optimizer_train"])
    retain_dev = expand(assignments["retain90_dev_utility_eval"])
    retain_final = expand(assignments["retain90_final_utility_eval"])
    holdout_dev = expand(assignments["holdout10_dev_utility_eval"])
    holdout_final = expand(assignments["holdout10_final_utility_eval"])
    excluded = set(assignments["retain90_excluded_conflicting_question_profiles"])
    if set(forget_dev) & set(forget_final) or sorted(forget_dev + forget_final) != list(range(20)):
        raise ValueError("Forget development/final blocks do not partition 0..19")
    if (set(retain_train) & set(retain_dev) or set(retain_train) & set(retain_final)
            or set(retain_dev) & set(retain_final) or excluded & (set(retain_train) | set(retain_dev) | set(retain_final))):
        raise ValueError("Retain training/evaluation blocks overlap or include excluded blocks")
    if set(retain_train) | set(retain_dev) | set(retain_final) | excluded != set(range(180)):
        raise ValueError("Retain profile assignment does not cover 0..179")
    if set(holdout_dev) & set(holdout_final) or sorted(holdout_dev + holdout_final) != list(range(20)):
        raise ValueError("Holdout development/final blocks do not partition 0..19")

    retain = rows_by_file["retain90.json"]
    answers_by_question: dict[str, set[str]] = {}
    for row in retain:
        answers_by_question.setdefault(row["question"], set()).add(row["answer"])
    collisions = sorted({i // 20 for i, row in enumerate(retain) if len(answers_by_question[row["question"]]) > 1})
    if collisions != sorted(excluded):
        raise ValueError(f"Question/answer collision profile mismatch: found {collisions}")
    forget = rows_by_file["forget10.json"]
    forget_perturbed = rows_by_file["forget10_perturbed.json"]
    if len(forget) != len(forget_perturbed) or any(
        (a["question"], a["answer"]) != (b["question"], b["answer"])
        for a, b in zip(forget, forget_perturbed)
    ):
        raise ValueError("Forget perturbed rows are not aligned with source order")

    idk_raw = idk_path.read_bytes()
    objective = protocol["objective"]
    if sha256(idk_raw) != objective["idk_response_sha256"]:
        raise ValueError("Pinned IDK response file hash mismatch")
    idk_lines = [line for line in idk_raw.decode("utf-8").splitlines() if line.strip()]
    if len(idk_lines) != int(objective["idk_response_count"]):
        raise ValueError("Pinned IDK response count mismatch")
    idk_rows = idk_lines
    if len(idk_rows) != 100 or any(not isinstance(x, str) or not x.strip() for x in idk_rows):
        raise ValueError("Official IDK response file must contain exactly 100 nonempty newline-delimited strings")

    counts = {
        "dev": {"forget10": len(forget_dev) * 20, "retain90_eval": len(retain_dev) * 20,
                "holdout10_eval": len(holdout_dev) * 20, "retain90_optimizer": len(retain_train) * 20},
        "final": {"forget10": len(forget_final) * 20, "retain90_eval": len(retain_final) * 20,
                  "holdout10_eval": len(holdout_final) * 20, "retain90_optimizer": len(retain_train) * 20},
    }
    expected = {
        "dev": {"forget10": 200, "retain90_eval": 400, "holdout10_eval": 200, "retain90_optimizer": 2760},
        "final": {"forget10": 200, "retain90_eval": 400, "holdout10_eval": 200, "retain90_optimizer": 2760},
    }
    if counts != expected:
        raise ValueError(f"Unexpected cohort sizes: {counts}")
    return {
        "status": "PASS_IDK_AP_PROTOCOL_DATA_AUDIT",
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": sha256(protocol_path.read_bytes()),
        "tofu_sha256": observed_hashes,
        "idk_responses_sha256": sha256(idk_raw),
        "idk_response_count": len(idk_rows),
        "profile_assignments": assignments,
        "excluded_retain_conflict_blocks": collisions,
        "cohort_counts": counts,
        "training_started": False,
        "model_weights_loaded": False,
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--protocol", type=Path, default=Path(__file__).with_name("idk_ap_lora_protocol.json"))
    parser.add_argument("--idk", type=Path, default=Path(__file__).with_name("idontknow.jsonl"))
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    result = validate(args.protocol, args.idk)
    encoded = json.dumps(result, indent=2) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(encoded, encoding="utf-8")
    print(encoded, end="")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/verify_objective.py
#!/usr/bin/env python3
"""Verify IDK+AP loss values and gradients against the pinned official code."""

from __future__ import annotations

import argparse
import copy
import hashlib
import importlib.util
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
import torch.nn.functional as F


IGNORE_INDEX = -100
UPSTREAM_COMMIT = "b622294965f8b40feb5383c9e93dbdb26a76b6b3"
UPSTREAM_LOSS_SHA256 = "078f64bb847bde0734747c00decf8cd06d44ca587e2c4eae2db01b12e7a2534f"
EXPECTED_PROTOCOL_SHA256 = "650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049"


class TinyCausalLM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(19, 9)
        self.head = torch.nn.Linear(9, 19, bias=False)

    def forward(self, input_ids, labels=None, attention_mask=None):
        logits = self.head(self.embedding(input_ids))
        loss = None
        if labels is not None:
            shifted_logits = logits[..., :-1, :].contiguous().float()
            shifted_labels = labels[..., 1:].contiguous().to(logits.device)
            loss = F.cross_entropy(
                shifted_logits.reshape(-1, shifted_logits.shape[-1]),
                shifted_labels.reshape(-1),
                ignore_index=IGNORE_INDEX,
                reduction="mean",
            )
        return SimpleNamespace(logits=logits, loss=loss)


def batch_sequence_nll(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = logits[..., :-1, :].contiguous().float()
    shifted_labels = labels[..., 1:].contiguous().to(logits.device)
    token_loss = F.cross_entropy(
        shifted_logits.transpose(-1, -2), shifted_labels,
        ignore_index=IGNORE_INDEX, reduction="none",
    )
    return token_loss.sum(dim=-1)


def candidate_get_loss(model, inputs, beta: float):
    forget_ids, forget_labels, forget_mask = inputs[2]
    forget = model(forget_ids, labels=forget_labels, attention_mask=forget_mask).loss
    retain_ids, retain_labels, retain_mask = inputs[1]
    idk_ids, idk_labels, idk_mask = inputs[3]
    answer_nll = batch_sequence_nll(model(retain_ids, attention_mask=retain_mask).logits, retain_labels)
    idk_nll = batch_sequence_nll(model(idk_ids, attention_mask=idk_mask).logits, idk_labels)
    ap = -F.logsigmoid(beta * (idk_nll - answer_nll)).mean() / beta
    return forget, ap


def max_grad_delta(left: torch.nn.Module, right: torch.nn.Module) -> float:
    values = []
    for a, b in zip(left.parameters(), right.parameters()):
        if a.grad is None or b.grad is None:
            if a.grad is not b.grad:
                return float("inf")
        else:
            values.append(float((a.grad - b.grad).abs().max()))
    return max(values, default=0.0)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--upstream-losses", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, default=Path(__file__).with_name("idk_ap_lora_protocol.json"))
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    expected_source = f"sail-sg/closer-look-LLM-unlearning@{UPSTREAM_COMMIT}"
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("IDK+AP protocol is not preregistered")
    if hashlib.sha256(args.protocol.read_bytes()).hexdigest() != EXPECTED_PROTOCOL_SHA256:
        raise RuntimeError("Locked protocol SHA-256 mismatch")
    if protocol["objective"].get("source_repository") + "@" + protocol["objective"].get("source_commit") != expected_source:
        raise RuntimeError("Protocol does not pin the expected IDK+AP source")
    source_bytes = args.upstream_losses.read_bytes()
    source_hash = hashlib.sha256(source_bytes).hexdigest()
    if source_hash != UPSTREAM_LOSS_SHA256 or source_hash != protocol["objective"]["loss_source_sha256"]:
        raise RuntimeError(f"Pinned official loss source hash mismatch: {source_hash}")
    spec = importlib.util.spec_from_file_location("official_idk_ap_losses", args.upstream_losses)
    official = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(official)

    # Variable response lengths verify that AP uses each response's summed token NLL.
    inputs = [
        None,
        (torch.tensor([[1, 2, 3, 4, 5, 6], [2, 3, 4, 5, 6, 7]]),
         torch.tensor([[-100, -100, -100, 4, 5, 6], [-100, -100, 4, 5, 6, 7]]),
         torch.ones((2, 6), dtype=torch.long)),
        (torch.tensor([[3, 4, 5, 6, 7, 8], [4, 5, 6, 7, 8, 9]]),
         torch.tensor([[-100, -100, 5, 6, 7, 8], [-100, -100, -100, -100, 8, 9]]),
         torch.ones((2, 6), dtype=torch.long)),
        (torch.tensor([[1, 3, 5, 7, 9, 11], [2, 4, 6, 8, 10, 12]]),
         torch.tensor([[-100, -100, -100, 7, 9, 11], [-100, -100, -100, 8, 10, 12]]),
         torch.ones((2, 6), dtype=torch.long)),
    ]
    beta = float(protocol["objective"]["beta"])
    torch.manual_seed(42)
    base = TinyCausalLM()
    official_model = copy.deepcopy(base)
    candidate_model = copy.deepcopy(base)
    official_forget, official_ap = official.get_loss(official_model, None, inputs, "IDK+AP", beta=beta)
    candidate_forget, candidate_ap = candidate_get_loss(candidate_model, inputs, beta)
    official_total = official_forget + official_ap
    candidate_total = candidate_forget + candidate_ap
    official_total.backward()
    candidate_total.backward()
    forget_diff = abs(float(official_forget.detach()) - float(candidate_forget.detach()))
    ap_diff = abs(float(official_ap.detach()) - float(candidate_ap.detach()))
    total_diff = abs(float(official_total.detach()) - float(candidate_total.detach()))
    grad_diff = max_grad_delta(official_model, candidate_model)
    passed = max(forget_diff, ap_diff, total_diff, grad_diff) <= 1e-6
    result = {
        "status": "PASS" if passed else "FAIL",
        "official_source": expected_source,
        "official_losses_py_sha256": source_hash,
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "torch_version": torch.__version__,
        "beta": beta,
        "official_forget_loss": float(official_forget.detach()),
        "candidate_forget_loss": float(candidate_forget.detach()),
        "forget_loss_abs_difference": forget_diff,
        "official_ap_loss": float(official_ap.detach()),
        "candidate_ap_loss": float(candidate_ap.detach()),
        "ap_loss_abs_difference": ap_diff,
        "total_loss_abs_difference": total_diff,
        "max_parameter_gradient_abs_difference": grad_diff,
        "batch_size": 2,
        "response_token_counts_vary": True,
        "loss_reduction": "forget IDK uses token-mean CE; AP uses mean of per-sequence summed-NLL preference losses",
    }
    encoded = json.dumps(result, indent=2) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(encoded, encoding="utf-8")
    print(encoded, end="")
    if not passed:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/train.py
#!/usr/bin/env python3
"""Resource-bounded LoRA adaptation of the official TOFU IDK+AP objective."""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import math
import os
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent))
from tofu_official_eval import CHECKPOINTS, DATASET_ID, DATASET_REVISION, encode_answer, get_bytes


IGNORE_INDEX = -100
MODEL_KEY = "llama_3_2_3b_full"
EXPECTED_PROTOCOL_SHA256 = "650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049"


def sha256(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()


def fetch_jsonl(filename: str) -> tuple[list[dict], str]:
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
    raw = get_bytes(url)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    return rows, sha256(raw)


def expand(ranges: list[list[int]]) -> list[int]:
    return [block for first, last in ranges for block in range(first, last + 1)]


def collate(rows: list[tuple[list[int], list[int]]], pad_id: int, device: torch.device):
    max_len = max(len(ids) for ids, _ in rows)
    input_ids = torch.full((len(rows), max_len), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), max_len), dtype=torch.long)
    labels = torch.full((len(rows), max_len), IGNORE_INDEX, dtype=torch.long)
    for i, (ids, row_labels) in enumerate(rows):
        size = len(ids)
        input_ids[i, :size] = torch.tensor(ids, dtype=torch.long)
        attention[i, :size] = 1
        labels[i, :size] = torch.tensor(row_labels, dtype=torch.long)
    return input_ids.to(device), attention.to(device), labels


def per_sequence_nll(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    shifted_logits = logits[..., :-1, :].contiguous().float()
    shifted_labels = labels[..., 1:].contiguous().to(logits.device)
    losses = F.cross_entropy(
        shifted_logits.transpose(-1, -2), shifted_labels,
        ignore_index=IGNORE_INDEX, reduction="none",
    )
    counts = (shifted_labels != IGNORE_INDEX).sum(dim=-1)
    if torch.any(counts == 0):
        raise ValueError("An IDK+AP example has no scored assistant tokens")
    return losses.sum(dim=-1), counts


def mean_token_ce(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = logits[..., :-1, :].contiguous().float()
    shifted_labels = labels[..., 1:].contiguous().to(logits.device)
    if not torch.any(shifted_labels != IGNORE_INDEX):
        raise ValueError("IDK forget target contains no assistant tokens")
    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.shape[-1]),
        shifted_labels.reshape(-1), ignore_index=IGNORE_INDEX, reduction="mean",
    )


def check_packages() -> dict[str, str]:
    expected = {"transformers": "5.22.1", "accelerate": "1.15.0", "peft": "0.21.2", "scipy": "1.16.3", "rouge-score": "0.1.2"}
    actual = {name: importlib.metadata.version(name) for name in expected}
    if actual != expected:
        raise RuntimeError(f"Pinned package mismatch: {actual}")
    try:
        torchao = importlib.metadata.version("torchao")
    except importlib.metadata.PackageNotFoundError:
        torchao = None
    if torchao is not None:
        raise RuntimeError(f"Unexpected torchao {torchao}; the locked run does not use it")
    return actual


def prepare_rows(protocol: dict, cohort: str, idk_path: Path):
    expected_hashes = protocol["data"]["sha256"]
    forget, forget_sha = fetch_jsonl("forget10.json")
    retain, retain_sha = fetch_jsonl("retain90.json")
    if forget_sha != expected_hashes["forget10.json"] or retain_sha != expected_hashes["retain90.json"]:
        raise RuntimeError("Pinned TOFU training file hash mismatch")
    if len(forget) != 400 or len(retain) != 3600:
        raise RuntimeError(f"Unexpected TOFU source row counts: forget={len(forget)}, retain={len(retain)}")
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    forget_blocks = expand(assignments[f"forget10_{cohort}_train_and_eval"])
    retain_blocks = expand(assignments["retain90_optimizer_train"])
    size = int(protocol["data"]["profile_block_size"])
    forget_indices = [i for i in range(len(forget)) if i // size in forget_blocks]
    retain_indices = [i for i in range(len(retain)) if i // size in retain_blocks]
    forget_selected = [forget[i] for i in forget_indices]
    retain_selected = [retain[i] for i in retain_indices]
    if len(forget_selected) != 200 or len(retain_selected) != 2760:
        raise RuntimeError(f"Wrong {cohort} training selection: forget={len(forget_selected)}, retain={len(retain_selected)}")
    idk_bytes = idk_path.read_bytes()
    idk_responses = [line.strip() for line in idk_bytes.decode("utf-8").splitlines() if line.strip()]
    if sha256(idk_bytes) != protocol["objective"]["idk_response_sha256"] or len(idk_responses) != 100:
        raise RuntimeError("Pinned IDK responses do not match the protocol")

    # Use a deterministic, fixed pairing. This adapts the official dataset's
    # index-seeded retain/IDK sampling to the leakage-screened optimizer pool.
    paired = []
    for index, f_row in enumerate(forget_selected):
        generator = torch.Generator(device="cpu").manual_seed(index)
        retain_local = (index + int(torch.randint(len(retain_selected), (1,), generator=generator).item())) % len(retain_selected)
        idk_local = int(torch.randint(len(idk_responses), (1,), generator=generator).item())
        paired.append({
            "forget_source_index": forget_indices[index],
            "retain_source_index": retain_indices[retain_local],
            "idk_index": idk_local,
            "forget": f_row,
            "retain": retain_selected[retain_local],
            "idk": idk_responses[idk_local],
        })
    return paired, {
        "forget10_sha256": forget_sha,
        "retain90_sha256": retain_sha,
        "idk_response_sha256": sha256(idk_bytes),
        "forget_source_indices": forget_indices,
        "retain_optimizer_source_indices": retain_indices,
        "forget_profile_blocks": forget_blocks,
        "retain_optimizer_profile_blocks": retain_blocks,
        "forget_rows": len(forget_selected),
        "retain_optimizer_rows": len(retain_selected),
        "pairing_assignments": [
            {"forget_source_index": p["forget_source_index"], "retain_source_index": p["retain_source_index"],
             "idk_response_index": p["idk_index"]} for p in paired
        ],
        "pairing_sha256": sha256(json.dumps([
            [p["forget_source_index"], p["retain_source_index"], p["idk_index"]] for p in paired
        ], separators=(",", ":")).encode()),
    }


def encode_pairs(tokenizer, paired, max_length: int):
    forget_idk = []
    retain_answer = []
    retain_idk = []
    for pair in paired:
        forget_idk.append(encode_answer(tokenizer, pair["forget"]["question"], pair["idk"]))
        retain_answer.append(encode_answer(tokenizer, pair["retain"]["question"], pair["retain"]["answer"]))
        retain_idk.append(encode_answer(tokenizer, pair["retain"]["question"], pair["idk"]))
    all_encoded = forget_idk + retain_answer + retain_idk
    lengths = [len(ids) for ids, _ in all_encoded]
    if max(lengths) > max_length:
        raise ValueError(f"Untruncated sequence exceeds locked {max_length}-token limit: {max(lengths)}")
    if any(not any(label != IGNORE_INDEX for label in labels[1:]) for _, labels in all_encoded):
        raise ValueError("A source sequence has an empty assistant response mask")
    return forget_idk, retain_answer, retain_idk, max(lengths)


def run_microbatch(model, triple, pad_id, input_device, beta, do_backward: bool, gradient_scale: float = 1.0):
    f_row, answer_row, idk_row = triple
    f_ids, f_mask, f_labels = collate([f_row], pad_id, input_device)
    r_ids, r_mask, r_labels = collate([answer_row], pad_id, input_device)
    i_ids, i_mask, i_labels = collate([idk_row], pad_id, input_device)

    f_logits = model(input_ids=f_ids, attention_mask=f_mask, use_cache=False).logits
    forget_loss = mean_token_ce(f_logits, f_labels)
    if not torch.isfinite(forget_loss):
        raise FloatingPointError("Non-finite forget IDK cross-entropy")
    if do_backward:
        (forget_loss * gradient_scale).backward()
    forget_value = float(forget_loss.detach().cpu())
    del f_logits, forget_loss

    r_logits = model(input_ids=r_ids, attention_mask=r_mask, use_cache=False).logits
    answer_nll, _ = per_sequence_nll(r_logits, r_labels)
    i_logits = model(input_ids=i_ids, attention_mask=i_mask, use_cache=False).logits
    idk_nll, _ = per_sequence_nll(i_logits, i_labels)
    ap_loss = -F.logsigmoid(beta * (idk_nll - answer_nll)).mean() / beta
    if not torch.isfinite(ap_loss):
        raise FloatingPointError("Non-finite answer-preservation preference loss")
    if do_backward:
        (ap_loss * gradient_scale).backward()
    ap_value = float(ap_loss.detach().cpu())
    total = forget_value + ap_value
    del r_logits, i_logits, answer_nll, idk_nll, ap_loss
    return forget_value, ap_value, total


def train(args) -> dict:
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    objective_check = json.loads(args.objective_check.read_text(encoding="utf-8"))
    data_audit = json.loads(args.data_audit.read_text(encoding="utf-8"))
    protocol_hash = sha256(args.protocol.read_bytes())
    if protocol_hash != EXPECTED_PROTOCOL_SHA256:
        raise RuntimeError("Locked protocol SHA-256 mismatch; use a new protocol version for design changes")
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN" or objective_check.get("status") != "PASS":
        raise RuntimeError("Protocol/objective gate failed before model loading")
    if data_audit.get("status") != "PASS_IDK_AP_PROTOCOL_DATA_AUDIT":
        raise RuntimeError("Pinned data/profile audit failed before model loading")
    if data_audit.get("protocol_sha256") != protocol_hash:
        raise RuntimeError("Data audit receipt belongs to a different protocol")
    if objective_check.get("protocol_sha256") != protocol_hash:
        raise RuntimeError("Objective-check receipt belongs to a different protocol")
    packages = check_packages()
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("This run requires exactly two Kaggle GPUs")
    gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
    if gpu_names.count("Tesla T4") != 2:
        raise RuntimeError(f"Locked hardware is Tesla T4 x2, found {gpu_names}")
    model_spec = CHECKPOINTS[MODEL_KEY]
    locked = protocol["model"]
    if (locked["repo_id"], locked["revision"]) != (model_spec["repo_id"], model_spec["revision"]):
        raise RuntimeError("Pinned model checkpoint does not match tofu_official_eval")
    if args.seed != int(protocol["training"]["seed"]):
        raise RuntimeError("Requested seed differs from preregistration")
    if args.cohort not in {"dev", "final"}:
        raise RuntimeError("Explicit dev/final cohort is required")

    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=False)
    receipt_path = output_dir / "training_receipt.json"
    trace_path = output_dir / "loss_trace.jsonl"
    receipt = {
        "status": "PREFLIGHT",
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": protocol_hash,
        "data_audit_sha256": sha256(args.data_audit.read_bytes()),
        "objective_equivalence": objective_check,
        "objective_equivalence_sha256": sha256(args.objective_check.read_bytes()),
        "training_script_sha256": sha256(Path(__file__).read_bytes()),
        "method": protocol["objective"],
        "training": protocol["training"],
        "cohort": args.cohort,
        "seed": args.seed,
        "checkpoint": model_spec,
        "gpus": gpu_names,
        "package_versions": {"torch": torch.__version__, **packages},
        "training_started": False,
        "adapter_saved": False,
    }
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
    started = time.monotonic()
    try:
        paired, selected = prepare_rows(protocol, args.cohort, args.idk)
        receipt["dataset"] = selected
        model_cache = Path(os.environ.get("TOFU_MODEL_CACHE_ROOT", "/kaggle/working/tofu_model_cache")) / MODEL_KEY
        model_cache.mkdir(parents=True, exist_ok=True)
        tokenizer = AutoTokenizer.from_pretrained(
            model_spec["repo_id"], revision=model_spec["revision"], cache_dir=str(model_cache), use_fast=True
        )
        if not tokenizer.is_fast:
            raise RuntimeError("Pinned tokenizer must be fast")
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        f_encoded, r_encoded, i_encoded, max_length = encode_pairs(
            tokenizer, paired, int(protocol["training"]["max_length"])
        )
        receipt["tokenization"] = {
            "max_length_observed": max_length,
            "truncated_examples": 0,
            "all_assistant_masks_nonempty": True,
            "encoding": "pinned Llama chat template from tofu_official_eval.encode_answer",
        }

        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = False
        model = AutoModelForCausalLM.from_pretrained(
            model_spec["repo_id"], revision=model_spec["revision"], cache_dir=str(model_cache), dtype=torch.float16,
            low_cpu_mem_usage=True, device_map="balanced",
            max_memory={0: "13GiB", 1: "13GiB", "cpu": "40GiB"},
            attn_implementation="sdpa",
        )
        device_map = getattr(model, "hf_device_map", {})
        offload = sorted({str(value) for value in device_map.values() if str(value).lower() in {"cpu", "disk"}})
        if offload:
            raise RuntimeError(f"CPU/disk offload is prohibited: {offload}")
        input_device = model.get_input_embeddings().weight.device
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        t = protocol["training"]
        lora = LoraConfig(
            r=int(t["lora_rank"]), lora_alpha=int(t["lora_alpha"]),
            lora_dropout=float(t["lora_dropout"]), target_modules=list(t["target_modules"]),
            bias="none", task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora)
        trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
        if not trainable or any(parameter.dtype != torch.float32 for parameter in trainable):
            raise RuntimeError("Expected FP32 LoRA-only trainable parameters")
        if any("lora_" not in name for name, parameter in model.named_parameters() if parameter.requires_grad):
            raise RuntimeError("A non-LoRA parameter is unexpectedly trainable")
        init_state = [parameter.detach().cpu().clone() for parameter in trainable]

        # A real-data one-step smoke validates forward/backward/optimizer behavior.
        # Restore the exact LoRA initialization so the diagnostic training starts clean.
        smoke_optimizer = torch.optim.AdamW(
            trainable, lr=float(t["learning_rate"]), betas=tuple(t["optimizer_betas"]),
            eps=float(t["optimizer_eps"]), weight_decay=float(t["weight_decay"]),
        )
        model.train()
        smoke_optimizer.zero_grad(set_to_none=True)
        beta = float(protocol["objective"]["beta"])
        smoke_triple = (f_encoded[0], r_encoded[0], i_encoded[0])
        smoke_losses = run_microbatch(model, smoke_triple, int(tokenizer.pad_token_id), input_device, beta, True)
        smoke_grad_norm = torch.nn.utils.clip_grad_norm_(trainable, float(t["gradient_clip_norm"]))
        if not torch.isfinite(smoke_grad_norm):
            raise FloatingPointError("Non-finite gradient norm in real-data smoke")
        smoke_optimizer.step()
        smoke_delta = max(float((parameter.detach().cpu() - initial).abs().max()) for parameter, initial in zip(trainable, init_state))
        if not math.isfinite(smoke_delta) or smoke_delta <= 0:
            raise RuntimeError("Smoke optimizer step did not change a LoRA parameter")
        with torch.no_grad():
            for parameter, initial in zip(trainable, init_state):
                parameter.copy_(initial.to(device=parameter.device, dtype=parameter.dtype))
        del smoke_optimizer
        for parameter in trainable:
            parameter.grad = None
        torch.cuda.empty_cache()
        receipt["runtime"] = {
            "device_map": {str(key): str(value) for key, value in device_map.items()},
            "cpu_or_disk_offload": False,
            "input_device": str(input_device),
            "base_dtype": "float16",
            "adapter_dtype": "float32",
            "attention": "sdpa",
            "real_data_smoke": {
                "status": "PASS_BOUNDED_REAL_DATA_SMOKE",
                "forget_idk_loss": smoke_losses[0],
                "retain_ap_loss": smoke_losses[1],
                "total_loss": smoke_losses[2],
                "max_abs_lora_update_before_reset": smoke_delta,
                "gradient_norm_before_clip": float(smoke_grad_norm.detach().cpu()),
                "initial_lora_restored_before_research_training": True,
            },
        }
        if any(not math.isfinite(value) for value in smoke_losses):
            raise FloatingPointError("Smoke losses were non-finite")

        epochs = int(t["epochs"])
        microbatch = int(t["microbatch_paired_examples"])
        accumulation = int(t["gradient_accumulation_steps"])
        if len(paired) % (microbatch * accumulation):
            raise RuntimeError("Locked cohort does not yield complete gradient-accumulation windows")
        steps_per_epoch = len(paired) // (microbatch * accumulation)
        total_steps = int(t["expected_optimizer_steps"])
        if steps_per_epoch * epochs != total_steps or total_steps != 125:
            raise RuntimeError("Locked optimizer schedule is internally inconsistent")
        optimizer = torch.optim.AdamW(
            trainable, lr=float(t["learning_rate"]), betas=tuple(t["optimizer_betas"]),
            eps=float(t["optimizer_eps"]), weight_decay=float(t["weight_decay"]),
        )
        scheduler = get_linear_schedule_with_warmup(optimizer, steps_per_epoch, total_steps)
        receipt["training_schedule"] = {
            "epochs": epochs,
            "microbatch_paired_examples": microbatch,
            "gradient_accumulation_steps": accumulation,
            "effective_paired_batch_size": microbatch * accumulation,
            "optimizer_steps_per_epoch": steps_per_epoch,
            "planned_optimizer_steps": total_steps,
            "planned_loss_trace_rows": total_steps * accumulation,
        }
        receipt["training_started"] = True
        receipt["status"] = "TRAINING"
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")

        rng = random.Random(args.seed)
        optimizer.zero_grad(set_to_none=True)
        global_step = 0
        with trace_path.open("w", encoding="utf-8") as trace:
            for epoch in range(epochs):
                order = list(range(len(paired)))
                rng.shuffle(order)
                pending = []
                for start in range(0, len(order), microbatch):
                    index = order[start]
                    losses = run_microbatch(
                        model,
                        (f_encoded[index], r_encoded[index], i_encoded[index]),
                        int(tokenizer.pad_token_id), input_device, beta, True, 1.0 / accumulation,
                    )
                    pending.append({
                        "epoch": epoch + 1,
                        "pair_index": index,
                        "forget_source_index": paired[index]["forget_source_index"],
                        "retain_source_index": paired[index]["retain_source_index"],
                        "idk_index": paired[index]["idk_index"],
                        "forget_idk_loss": losses[0],
                        "retain_ap_loss": losses[1],
                        "combined_loss": losses[2],
                    })
                    if len(pending) == accumulation:
                        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, float(t["gradient_clip_norm"]))
                        if not torch.isfinite(grad_norm):
                            raise FloatingPointError("Non-finite gradient norm")
                        optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)
                        global_step += 1
                        lr_now = scheduler.get_last_lr()[0]
                        for record in pending:
                            record.update({
                                "global_step": global_step,
                                "learning_rate": lr_now,
                                "gradient_norm_preclip": float(grad_norm.detach().cpu()),
                                "clip_threshold": float(t["gradient_clip_norm"]),
                            })
                            trace.write(json.dumps(record, sort_keys=True) + "\n")
                        trace.flush()
                        pending.clear()
                        if global_step % 10 == 0:
                            print(json.dumps({"epoch": epoch + 1, "global_step": global_step, "planned_steps": total_steps,
                                              "lr": lr_now, "last_microbatch_loss": losses[2]}), flush=True)
                if pending:
                    raise RuntimeError("Unexpected incomplete gradient accumulation window")
        if global_step != total_steps:
            raise RuntimeError(f"Expected {total_steps} optimizer updates, received {global_step}")

        adapter_dir = output_dir / "adapter"
        model.save_pretrained(adapter_dir, safe_serialization=True)
        tokenizer.save_pretrained(adapter_dir)
        receipt.update({
            "status": "TRAINING_COMPLETE",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "optimizer_steps": global_step,
            "trainable_parameters": sum(parameter.numel() for parameter in trainable),
            "all_parameters": sum(parameter.numel() for parameter in model.parameters()),
            "adapter_saved": True,
            "adapter_files_sha256": {path.name: sha256(path.read_bytes()) for path in sorted(adapter_dir.iterdir()) if path.is_file()},
            "loss_trace_sha256": sha256(trace_path.read_bytes()),
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        print(json.dumps(receipt, indent=2), flush=True)
        return receipt
    except Exception as error:
        receipt.update({
            "status": "ERROR",
            "error": f"{type(error).__name__}: {error}",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        raise


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--objective-check", type=Path, required=True)
    parser.add_argument("--data-audit", type=Path, required=True)
    parser.add_argument("--idk", type=Path, required=True)
    parser.add_argument("--cohort", choices=("dev", "final"), required=True)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    train(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/evaluate.py
#!/usr/bin/env python3
"""Evaluate the IDK+AP adapter on locked profile blocks and apply fixed gates."""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import random
import re
import sys
import unicodedata
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent))
from evaluate_npo_profile_cohort import expand, make_rows
from tofu_official_eval import CHECKPOINTS, DATASET_REVISION, evaluate_checkpoint, load_data


IDK_MARKER = re.compile(
    r"\b(?:i\s+(?:do\s+not|don't)\s+know|i\s+(?:am\s+not|m\s+not)\s+(?:sure|certain)|"
    r"i\s+(?:do\s+not|don't)\s+have\s+(?:the\s+)?(?:answer|information)|"
    r"i\s+cannot\s+answer|i\s+can't\s+answer|not\s+enough\s+information|unable\s+to\s+answer)\b",
    flags=re.IGNORECASE,
)
BOOTSTRAP_DRAWS = 5000
EXPECTED_PROTOCOL_SHA256 = "650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049"


def sha256(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()


def normalize_text(text: str) -> str:
    value = unicodedata.normalize("NFKC", text).casefold().strip()
    return re.sub(r"[^\w]+", " ", value, flags=re.UNICODE).strip()


def select_eval_sets(data: dict[str, list[dict]], protocol: dict, cohort: str) -> dict[str, list[dict]]:
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    if cohort == "dev":
        forget_key = "forget10_dev_train_and_eval"
        retain_key = "retain90_dev_utility_eval"
        holdout_key = "holdout10_dev_utility_eval"
    else:
        forget_key = "forget10_final_train_and_eval"
        retain_key = "retain90_final_utility_eval"
        holdout_key = "holdout10_final_utility_eval"
    return {
        "forget10": make_rows(data, "forget10", expand(assignments[forget_key])),
        "retain90": make_rows(data, "retain90", expand(assignments[retain_key])),
        "holdout10": make_rows(data, "holdout10", expand(assignments[holdout_key])),
    }


def paired_cluster_delta(base_rows: list[dict], candidate_rows: list[dict], field: str, seed: int) -> dict:
    base_by_index = {int(row["source_row_index"]): row for row in base_rows}
    groups: dict[int, list[float]] = {}
    for row in candidate_rows:
        source_index = int(row["source_row_index"])
        base = base_by_index.get(source_index)
        if base is None:
            raise ValueError(f"Unmatched source index {source_index} in {field}")
        groups.setdefault(int(row["profile_block"]), []).append(float(row[field]) - float(base[field]))
    if not groups:
        raise ValueError(f"No matched examples for {field}")
    block_ids = sorted(groups)
    point = sum(sum(values) for values in groups.values()) / sum(map(len, groups.values()))
    rng = random.Random(seed)
    draws = []
    for _ in range(BOOTSTRAP_DRAWS):
        sampled_blocks = [rng.choice(block_ids) for _ in block_ids]
        values = [value for block in sampled_blocks for value in groups[block]]
        draws.append(sum(values) / len(values))
    draws.sort()
    return {
        "n_questions": sum(map(len, groups.values())),
        "n_profile_blocks": len(groups),
        "mean_delta_candidate_minus_full": point,
        "ci95": [draws[int(0.025 * (BOOTSTRAP_DRAWS - 1))], draws[int(0.975 * (BOOTSTRAP_DRAWS - 1))]],
        "bootstrap_draws": BOOTSTRAP_DRAWS,
        "bootstrap_seed": seed,
    }


def summarize_and_gate(reference: dict, candidate: dict, protocol: dict) -> dict:
    margin = float(protocol["evaluation"]["noninferiority_margin"])
    rules = {
        "forget10.truth_ratio_score": ("forget10", "truth_ratio_score", "lower_zero"),
        "forget10.answer_probability": ("forget10", "answer_probability", "upper_zero"),
        "retain90.answer_probability": ("retain90", "answer_probability", "lower_margin"),
        "retain90.rougeL_recall": ("retain90", "rougeL_recall", "lower_margin"),
        "holdout10.answer_probability": ("holdout10", "answer_probability", "lower_margin"),
    }
    gates = {}
    for name, (split, metric, rule) in rules.items():
        delta = paired_cluster_delta(reference["per_example"][split], candidate["per_example"][split], metric, 42)
        low, high = delta["ci95"]
        passed = low > 0 if rule == "lower_zero" else high < 0 if rule == "upper_zero" else low > -margin
        gates[name] = {
            **delta,
            "metric": metric,
            "rule": rule,
            "threshold": 0.0 if rule.endswith("zero") else -margin,
            "pass": bool(passed),
        }
    all_pass = all(gate["pass"] for gate in gates.values())
    return {
        "status": "PASS_DIAGNOSTIC_GATES" if all_pass else "FAIL_DIAGNOSTIC_GATES",
        "all_gates_pass": all_pass,
        "interpretation": protocol["evaluation"]["research_claim_limit"],
        "gates": gates,
        "bootstrap": {"unit": "profile block", "draws": BOOTSTRAP_DRAWS, "seed": 42},
        "noninferiority_margin": margin,
    }


def add_idk_descriptives(result: dict, idk_path: Path) -> dict:
    responses = [line.strip() for line in idk_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    normalized = {normalize_text(answer) for answer in responses}
    rows = result["per_example"].get("forget10", [])
    hits = 0
    exact = 0
    for row in rows:
        text = str(row.get("generation", ""))
        row["idk_marker_match"] = bool(IDK_MARKER.search(text))
        row["exact_official_idk_match"] = normalize_text(text) in normalized
        hits += int(row["idk_marker_match"])
        exact += int(row["exact_official_idk_match"])
    result["idk_generation_diagnostics"] = {
        "n_forget_questions": len(rows),
        "idk_marker_matches": hits,
        "idk_marker_rate": hits / len(rows) if rows else None,
        "exact_official_idk_matches": exact,
        "exact_official_idk_rate": exact / len(rows) if rows else None,
        "marker_rule": IDK_MARKER.pattern,
        "limitation": "Descriptive string-matching signals only; not semantic refusal classification or a research gate.",
    }
    return result


def verify_reference_support(support_dir: Path, protocol: dict) -> Path:
    manifest_path = support_dir / "reference_support_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = protocol["reference_support"]
    result_path = support_dir / expected["reference_file"]
    if manifest.get("status") != "PASS_SOURCE_ARCHIVE_AND_PRIOR_VERIFICATION":
        raise ValueError("Reference-support manifest status did not pass")
    if manifest.get("source_archive_sha256") != expected["source_archive_sha256"]:
        raise ValueError("Reference source archive differs from the locked protocol")
    entry = manifest.get("files", {}).get("reference_eval/full_reference_profile_eval.json", {})
    if entry.get("sha256") != expected["reference_file_sha256"] or sha256(result_path.read_bytes()) != expected["reference_file_sha256"]:
        raise ValueError("Full-reference evaluation hash mismatch")
    reference = json.loads(result_path.read_text(encoding="utf-8"))
    if reference.get("status") != "PASS_PROFILE_COHORT_EVAL":
        raise ValueError("Frozen-checkpoint reference evaluation did not pass")
    model = reference.get("model", {})
    if (model.get("repo_id"), model.get("revision")) != (expected["reference_checkpoint_repo"], expected["reference_checkpoint_revision"]):
        raise ValueError("Frozen reference checkpoint does not match the protocol")
    if reference.get("evaluation", {}).get("dataset_revision") != protocol["data"]["revision"]:
        raise ValueError("Frozen reference was scored on a different TOFU revision")
    return result_path


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--cohort", choices=("dev", "final"), required=True)
    parser.add_argument("--adapter-dir", type=Path, required=True)
    parser.add_argument("--training-receipt", type=Path, required=True)
    parser.add_argument("--reference-support-dir", type=Path, required=True)
    parser.add_argument("--idk", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    receipt = json.loads(args.training_receipt.read_text(encoding="utf-8"))
    if sha256(args.protocol.read_bytes()) != EXPECTED_PROTOCOL_SHA256:
        raise ValueError("Locked protocol SHA-256 mismatch")
    if receipt.get("status") != "TRAINING_COMPLETE" or not receipt.get("adapter_saved"):
        raise ValueError("No completed training receipt is available")
    if receipt.get("protocol_sha256") != sha256(args.protocol.read_bytes()) or receipt.get("cohort") != args.cohort:
        raise ValueError("Training receipt does not match the locked protocol/cohort")
    expected_adapter_hashes = receipt.get("adapter_files_sha256", {})
    actual_adapter_hashes = {path.name: sha256(path.read_bytes()) for path in sorted(args.adapter_dir.iterdir()) if path.is_file()}
    if not actual_adapter_hashes or expected_adapter_hashes != actual_adapter_hashes:
        raise ValueError("Adapter files do not match their training receipt")

    data, hashes = load_data()
    for filename, expected_hash in protocol["data"]["sha256"].items():
        if hashes.get(filename) != expected_hash:
            raise ValueError(f"Pinned TOFU evaluation source hash mismatch: {filename}")
    eval_sets = select_eval_sets(data, protocol, args.cohort)
    reference_path = verify_reference_support(args.reference_support_dir, protocol)
    reference = json.loads(reference_path.read_text(encoding="utf-8"))
    selected_indices = {split: {int(row["source_row_index"]) for row in rows} for split, rows in eval_sets.items()}
    ref_subset = {
        split: [row for row in reference["per_example"][split] if int(row["source_row_index"]) in selected_indices[split]]
        for split in eval_sets
    }
    for split, rows in ref_subset.items():
        if {int(row["source_row_index"]) for row in rows} != selected_indices[split]:
            raise ValueError(f"Frozen reference does not cover the exact {args.cohort} {split} rows")
        by_index = {int(row["source_row_index"]): row for row in rows}
        for current in eval_sets[split]:
            prior = by_index[int(current["source_row_index"])]
            if hashlib.sha256(current["question"].encode()).hexdigest() != prior.get("question_sha256"):
                raise ValueError(f"Frozen reference question mismatch at {split}:{current['source_row_index']}")
    reference_subset = {**reference, "per_example": ref_subset}

    args.output_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TOFU_KEEP_MODEL_CACHE"] = "1"
    result = evaluate_checkpoint(
        "llama_3_2_3b_full", args.output_dir, eval_sets, hashes,
        score_batch_size=1, generation_batch_size=1, max_memory_per_gpu_gib=12.5,
        adapter_path=args.adapter_dir, result_key=f"{args.cohort}_idk_ap",
        adapter_seed=int(protocol["training"]["seed"]), profile_cohort_mode=True,
    )
    if result.get("status") != "PASS_PROFILE_COHORT_EVAL":
        raise RuntimeError("IDK+AP adapter evaluation did not complete")
    result["model"]["label"] = "Llama 3.2 3B full IDK+AP LoRA diagnostic seed 42"
    if result["model"].get("adapter"):
        result["model"]["adapter"]["method"] = "IDK+AP LoRA adaptation"
    add_idk_descriptives(result, args.idk)
    result_path = args.output_dir / f"{args.cohort}_idk_ap_profile_eval.json"
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    comparison = summarize_and_gate(reference_subset, result, protocol)
    comparison.update({
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": sha256(args.protocol.read_bytes()),
        "cohort": args.cohort,
        "model_revision": protocol["model"]["revision"],
        "dataset_revision": DATASET_REVISION,
        "training_receipt_sha256": sha256(args.training_receipt.read_bytes()),
        "evaluation_receipt": result_path.name,
        "candidate_idk_diagnostics": result["idk_generation_diagnostics"],
    })
    comparison_path = args.output_dir / f"{args.cohort}_idk_ap_comparison.json"
    comparison_path.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(comparison, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/run.py
#!/usr/bin/env python3
"""Run the preregistered IDK+AP development diagnostic and fixed stop rule."""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path


HERE = Path(__file__).resolve().parent
ROOT = HERE.parent


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2) + "\n", encoding="utf-8")


def run_stage(name: str, command: list[str], root: Path, env: dict | None = None) -> tuple[int, Path]:
    logs = root / "logs"
    logs.mkdir(exist_ok=True)
    path = logs / f"{name}.log"
    started = time.monotonic()
    print(json.dumps({"stage": name, "status": "STARTED", "command": command}), flush=True)
    with path.open("w", encoding="utf-8") as log:
        process = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True, env=env)
    print(json.dumps({"stage": name, "status": "PASS" if process.returncode == 0 else "ERROR",
                      "returncode": process.returncode, "elapsed_seconds": round(time.monotonic() - started, 2),
                      "log": str(path)}), flush=True)
    if process.returncode:
        tail = path.read_text(encoding="utf-8", errors="replace")[-5000:]
        print(tail, flush=True)
    return process.returncode, path


def zip_run(root: Path, zip_path: Path) -> dict:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(root.rglob("*")):
            if path.is_file() and path != zip_path:
                archive.write(path, path.relative_to(root.parent))
    with zipfile.ZipFile(zip_path) as archive:
        bad = archive.testzip()
    if bad:
        raise RuntimeError(f"Artifact ZIP CRC failed at {bad}")
    return {"path": str(zip_path), "bytes": zip_path.stat().st_size, "sha256": sha256(zip_path), "crc": "PASS"}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--protocol", type=Path, default=HERE / "idk_ap_lora_protocol.json")
    parser.add_argument("--idk", type=Path, default=HERE / "idontknow.jsonl")
    parser.add_argument("--upstream-repo", type=Path, required=True)
    parser.add_argument("--reference-support-dir", type=Path, required=True)
    parser.add_argument("--root", type=Path, required=True)
    parser.add_argument("--artifact-zip", type=Path, required=True)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    protocol_hash = sha256(args.protocol)
    root = args.root
    root.mkdir(parents=True, exist_ok=False)
    upstream_commit = subprocess.check_output(
        ["git", "-C", str(args.upstream_repo), "rev-parse", "HEAD"], text=True
    ).strip()
    expected_commit = protocol["objective"]["source_commit"]
    if upstream_commit != expected_commit:
        raise RuntimeError(f"Official IDK+AP source checkout is not pinned: {upstream_commit}")
    upstream_losses = args.upstream_repo / protocol["objective"]["loss_source_path"]
    if sha256(upstream_losses) != protocol["objective"]["loss_source_sha256"]:
        raise RuntimeError("Pinned official objective source file hash mismatch")
    snapshot_sources = [
        args.protocol, args.idk, *HERE.glob("*.py"), HERE / "requirements.txt",
        ROOT / "tofu_official_eval.py", ROOT / "evaluate_npo_profile_cohort.py", upstream_losses,
    ]
    for source in snapshot_sources:
        if source.is_file():
            name = "official_losses_pinned.py" if source == upstream_losses else source.name
            destination = root / "source_snapshot" / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    receipt = {
        "status": "RUNNING",
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": protocol_hash,
        "upstream_source_commit": upstream_commit,
        "upstream_losses_sha256": sha256(upstream_losses),
        "kaggle_gpu_names": [],
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "training_started": False,
        "development_training_started": False,
        "final_training_started": False,
        "development_gate_status": None,
    }
    receipt_path = root / "run_receipt.json"
    write_json(root / "idk_ap_lora_protocol.json", protocol)
    write_json(root / "source_provenance.json", {
        "protocol_sha256": protocol_hash,
        "idk_response_file_sha256": sha256(args.idk),
        "upstream_commit": upstream_commit,
        "upstream_loss_sha256": sha256(upstream_losses),
        "source_snapshot_sha256": {
            path.name: sha256(path) for path in sorted((root / "source_snapshot").iterdir()) if path.is_file()
        },
    })
    write_json(receipt_path, receipt)
    try:
        audit_path = root / "idk_ap_data_audit.json"
        command = [sys.executable, str(HERE / "validate_protocol.py"), "--protocol", str(args.protocol), "--idk", str(args.idk), "--output", str(audit_path)]
        if run_stage("pinned-data-profile-audit", command, root)[0]:
            raise RuntimeError("Pinned data/profile audit failed")
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if audit.get("status") != "PASS_IDK_AP_PROTOCOL_DATA_AUDIT":
            raise RuntimeError("Pinned data/profile audit receipt failed")
        objective_path = root / "idk_ap_objective_equivalence.json"
        command = [sys.executable, str(HERE / "verify_objective.py"), "--upstream-losses", str(upstream_losses),
                   "--protocol", str(args.protocol), "--output", str(objective_path)]
        if run_stage("official-objective-value-gradient-equivalence", command, root)[0]:
            raise RuntimeError("Official objective equivalence check failed")
        objective = json.loads(objective_path.read_text(encoding="utf-8"))
        if objective.get("status") != "PASS":
            raise RuntimeError("Official objective equivalence receipt failed")

        from evaluate import verify_reference_support
        reference_path = verify_reference_support(args.reference_support_dir, protocol)
        reference_copy = root / "reference_support" / "full_reference_profile_eval.json"
        reference_copy.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(reference_path, reference_copy)
        shutil.copy2(args.reference_support_dir / "reference_support_manifest.json", root / "reference_support" / "reference_support_manifest.json")

        data_audit_path = audit_path
        dev_dir = root / "dev"
        receipt["training_started"] = True
        receipt["development_training_started"] = True
        write_json(receipt_path, receipt)
        command = [sys.executable, str(HERE / "train.py"), "--protocol", str(args.protocol),
                   "--objective-check", str(objective_path), "--data-audit", str(data_audit_path),
                   "--idk", str(args.idk), "--cohort", "dev", "--seed", str(protocol["training"]["seed"]),
                   "--output-dir", str(dev_dir / "training")]
        if run_stage("dev-idk-ap-training", command, root)[0]:
            raise RuntimeError("Development IDK+AP training failed")
        train_receipt_path = dev_dir / "training" / "training_receipt.json"
        train_receipt = json.loads(train_receipt_path.read_text(encoding="utf-8"))
        if train_receipt.get("status") != "TRAINING_COMPLETE" or train_receipt.get("optimizer_steps") != protocol["training"]["expected_optimizer_steps"]:
            raise RuntimeError("Development training did not complete the locked schedule")
        command = [sys.executable, str(HERE / "evaluate.py"), "--protocol", str(args.protocol),
                   "--cohort", "dev", "--adapter-dir", str(dev_dir / "training" / "adapter"),
                   "--training-receipt", str(train_receipt_path), "--reference-support-dir", str(args.reference_support_dir),
                   "--idk", str(args.idk), "--output-dir", str(dev_dir / "evaluation")]
        if run_stage("dev-idk-ap-evaluation-and-gates", command, root)[0]:
            raise RuntimeError("Development evaluation or gate calculation failed")
        dev_comparison_path = dev_dir / "evaluation" / "dev_idk_ap_comparison.json"
        dev_comparison = json.loads(dev_comparison_path.read_text(encoding="utf-8"))
        receipt["development_gate_status"] = dev_comparison["status"]
        receipt["development_comparison"] = str(dev_comparison_path.relative_to(root))
        write_json(receipt_path, receipt)
        if not dev_comparison.get("all_gates_pass"):
            receipt.update({
                "status": "DEV_GATES_FAILED_STOP",
                "final_training_started": False,
                "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            write_json(receipt_path, receipt)
            return

        final_dir = root / "final"
        receipt["final_training_started"] = True
        write_json(receipt_path, receipt)
        command = [sys.executable, str(HERE / "train.py"), "--protocol", str(args.protocol),
                   "--objective-check", str(objective_path), "--data-audit", str(data_audit_path),
                   "--idk", str(args.idk), "--cohort", "final", "--seed", str(protocol["training"]["seed"]),
                   "--output-dir", str(final_dir / "training")]
        if run_stage("final-idk-ap-training", command, root)[0]:
            raise RuntimeError("Final IDK+AP training failed")
        final_receipt_path = final_dir / "training" / "training_receipt.json"
        final_receipt = json.loads(final_receipt_path.read_text(encoding="utf-8"))
        if final_receipt.get("status") != "TRAINING_COMPLETE" or final_receipt.get("optimizer_steps") != protocol["training"]["expected_optimizer_steps"]:
            raise RuntimeError("Final training did not complete the locked schedule")
        command = [sys.executable, str(HERE / "evaluate.py"), "--protocol", str(args.protocol),
                   "--cohort", "final", "--adapter-dir", str(final_dir / "training" / "adapter"),
                   "--training-receipt", str(final_receipt_path), "--reference-support-dir", str(args.reference_support_dir),
                   "--idk", str(args.idk), "--output-dir", str(final_dir / "evaluation")]
        if run_stage("final-idk-ap-evaluation-and-gates", command, root)[0]:
            raise RuntimeError("Final evaluation or gate calculation failed")
        final_comparison_path = final_dir / "evaluation" / "final_idk_ap_comparison.json"
        final_comparison = json.loads(final_comparison_path.read_text(encoding="utf-8"))
        receipt.update({
            "status": final_comparison["status"],
            "final_comparison": str(final_comparison_path.relative_to(root)),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        write_json(receipt_path, receipt)
    except Exception as error:
        receipt.update({
            "status": "ERROR",
            "error": f"{type(error).__name__}: {error}",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        })
        write_json(receipt_path, receipt)
        raise
    finally:
        zipped = zip_run(root, args.artifact_zip)
        archive_manifest = args.artifact_zip.with_suffix(args.artifact_zip.suffix + ".manifest.json")
        write_json(archive_manifest, {"artifact_archive": zipped, "run_status": receipt.get("status"),
                                     "protocol_id": protocol["protocol_id"], "protocol_sha256": protocol_hash})
        cache_root = Path(os.environ.get("TOFU_MODEL_CACHE_ROOT", ""))
        if str(cache_root) not in {"", "."}:
            shutil.rmtree(cache_root, ignore_errors=True)
        print(json.dumps({"run_receipt": receipt, "artifact_archive_manifest": str(archive_manifest),
                          "artifact_archive": zipped}, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/verify_run.py
#!/usr/bin/env python3
"""Independent artifact, training-trace, coverage, and gate recomputation."""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import urllib.request
import zipfile
from pathlib import Path


EXPECTED_PROTOCOL_SHA256 = "650af06eed1d254cfdd2b47714299686925cea47b5b2675edb7c07674b01c049"
PROFILE_SIZE = 20
DRAW_COUNT = 5000


def sha256(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def check_source_files(protocol: dict, protocol_path: Path, root: Path) -> list[str]:
    problems = []
    files = protocol["data"]["sha256"]
    for filename, expected in files.items():
        url = f"https://huggingface.co/datasets/locuslab/TOFU/resolve/{protocol['data']['revision']}/{filename}"
        request = urllib.request.Request(url, headers={"User-Agent": "fsds-idk-ap-independent-verifier/1.0"})
        with urllib.request.urlopen(request, timeout=60) as response:
            raw = response.read()
        if sha256(raw) != expected:
            problems.append(f"Pinned source hash mismatch: {filename}")
        rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
        if len(rows) != int(protocol["data"]["expected_rows"][filename]):
            problems.append(f"Pinned row count mismatch: {filename}")
    idk = root / "source_snapshot" / "idontknow.jsonl"
    if not idk.is_file() or sha256(idk.read_bytes()) != protocol["objective"]["idk_response_sha256"]:
        problems.append("Pinned IDK-response file is missing or has the wrong hash")
    if sha256(protocol_path.read_bytes()) != EXPECTED_PROTOCOL_SHA256:
        problems.append("Protocol digest differs from the locked preregistration")
    provenance = read_json(root / "source_provenance.json")
    for filename, expected_hash in provenance.get("source_snapshot_sha256", {}).items():
        path = root / "source_snapshot" / filename
        if not path.is_file() or sha256(path.read_bytes()) != expected_hash:
            problems.append(f"Source snapshot hash mismatch: {filename}")
    return problems


def expected_indices(ranges: list[list[int]]) -> set[int]:
    blocks = [block for first, last in ranges for block in range(first, last + 1)]
    return {block * PROFILE_SIZE + offset for block in blocks for offset in range(PROFILE_SIZE)}


def verify_rows(rows: list[dict], split: str, ranges: list[list[int]]) -> list[str]:
    expected = expected_indices(ranges)
    actual = {int(row["source_row_index"]) for row in rows}
    problems = []
    if len(rows) != len(expected) or actual != expected:
        problems.append(f"{split}: source rows do not match the locked profile assignment")
    if len(actual) != len(rows):
        problems.append(f"{split}: duplicate source-row IDs")
    expected_blocks = {index // PROFILE_SIZE for index in expected}
    actual_blocks = {int(row["profile_block"]) for row in rows}
    if expected_blocks != actual_blocks:
        problems.append(f"{split}: profile blocks do not match the locked assignment")
    for row in rows:
        for field in ("answer_probability", "answer_token_loss", "rougeL_recall"):
            try:
                value = float(row[field])
            except (KeyError, TypeError, ValueError):
                problems.append(f"{split}: missing/non-numeric {field}")
                continue
            if not math.isfinite(value):
                problems.append(f"{split}: non-finite {field}")
        if not 0 <= float(row["answer_probability"]) <= 1:
            problems.append(f"{split}: answer_probability outside [0,1]")
        if not 0 <= float(row["rougeL_recall"]) <= 1:
            problems.append(f"{split}: ROUGE-L outside [0,1]")
        if not isinstance(row.get("generation"), str) or not isinstance(row.get("reference_answer"), str):
            problems.append(f"{split}: generation/reference text missing")
        if abs(float(row["answer_probability"]) - math.exp(-float(row["answer_token_loss"]))) > 2e-8:
            problems.append(f"{split}: answer_probability does not match token loss")
        if split == "forget10":
            wrong = row.get("perturbed_answer_token_losses")
            if not wrong or any(not math.isfinite(float(value)) for value in wrong):
                problems.append("forget10: perturbed-answer NLLs missing or non-finite")
            else:
                wrong_probability = math.exp(-sum(map(float, wrong)) / len(wrong))
                if row.get("paraphrased_answer_token_loss") is not None:
                    correct_probability = math.exp(-float(row["paraphrased_answer_token_loss"]))
                    ratio = wrong_probability / (correct_probability + 1e-10)
                    score = min(ratio, 1.0 / (ratio + 1e-10))
                else:
                    ratio = wrong_probability / (float(row["answer_probability"]) + 1e-10)
                    score = max(0.0, 1.0 - ratio)
                if abs(ratio - float(row.get("truth_ratio", math.nan))) > 2e-8:
                    problems.append("forget10: Truth Ratio recomputation mismatch")
                if abs(score - float(row.get("truth_ratio_score", math.nan))) > 2e-8:
                    problems.append("forget10: Truth Ratio score recomputation mismatch")
    return problems


def paired_bootstrap(base_rows: list[dict], candidate_rows: list[dict], metric: str) -> dict:
    base = {int(row["source_row_index"]): row for row in base_rows}
    grouped: dict[int, list[float]] = {}
    for row in candidate_rows:
        index = int(row["source_row_index"])
        if index not in base:
            raise ValueError(f"Unmatched source index {index} in {metric}")
        grouped.setdefault(int(row["profile_block"]), []).append(float(row[metric]) - float(base[index][metric]))
    block_ids = sorted(grouped)
    point = sum(sum(v) for v in grouped.values()) / sum(map(len, grouped.values()))
    rng = random.Random(42)
    samples = []
    for _ in range(DRAW_COUNT):
        selected = [rng.choice(block_ids) for _ in block_ids]
        values = [x for block in selected for x in grouped[block]]
        samples.append(sum(values) / len(values))
    samples.sort()
    return {
        "mean_delta_candidate_minus_full": point,
        "ci95": [samples[int(0.025 * (DRAW_COUNT - 1))], samples[int(0.975 * (DRAW_COUNT - 1))]],
        "n_questions": sum(map(len, grouped.values())),
        "n_profile_blocks": len(grouped),
    }


def verify_comparison(reference: dict, candidate: dict, comparison: dict, protocol: dict, cohort: str) -> list[str]:
    ranges = protocol["data"]["profile_assignment_ranges_inclusive"]
    split_ranges = {
        "forget10": ranges[f"forget10_{cohort}_train_and_eval"],
        "retain90": ranges[f"retain90_{cohort}_utility_eval"],
        "holdout10": ranges[f"holdout10_{cohort}_utility_eval"],
    }
    problems = []
    for split in split_ranges:
        problems += verify_rows(candidate["per_example"].get(split, []), split, split_ranges[split])
    expected_indices = {split: expected_indices_for_ranges(value) for split, value in split_ranges.items()}
    reference_by_split = {}
    for split in split_ranges:
        reference_rows = reference["per_example"].get(split, [])
        reference_by_split[split] = [row for row in reference_rows if int(row["source_row_index"]) in expected_indices[split]]
        if {int(row["source_row_index"]) for row in reference_by_split[split]} != expected_indices[split]:
            problems.append(f"Reference coverage mismatch: {split}")
        candidate_by_index = {int(row["source_row_index"]): row for row in candidate["per_example"][split]}
        for row in reference_by_split[split]:
            candidate_row = candidate_by_index.get(int(row["source_row_index"]))
            if candidate_row is None or candidate_row.get("question_sha256") != row.get("question_sha256"):
                problems.append(f"Paired question mismatch: {split}:{row.get('source_row_index')}")

    gate_specs = {
        "forget10.truth_ratio_score": ("forget10", "truth_ratio_score", "low_zero"),
        "forget10.answer_probability": ("forget10", "answer_probability", "high_zero"),
        "retain90.answer_probability": ("retain90", "answer_probability", "low_margin"),
        "retain90.rougeL_recall": ("retain90", "rougeL_recall", "low_margin"),
        "holdout10.answer_probability": ("holdout10", "answer_probability", "low_margin"),
    }
    margin = float(protocol["evaluation"]["noninferiority_margin"])
    expected_gates = {}
    for name, (split, metric, rule) in gate_specs.items():
        stats = paired_bootstrap(reference_by_split[split], candidate["per_example"][split], metric)
        low, high = stats["ci95"]
        passed = low > 0 if rule == "low_zero" else high < 0 if rule == "high_zero" else low > -margin
        expected_gates[name] = {**stats, "pass": passed}
    recorded = comparison.get("gates", {})
    if set(recorded) != set(expected_gates):
        problems.append("Recorded metric gate set mismatch")
    for name, expected in expected_gates.items():
        observed = recorded.get(name, {})
        for key in ("n_questions", "n_profile_blocks", "pass"):
            if observed.get(key) != expected[key]:
                problems.append(f"Gate mismatch for {name}: {key}")
        for key in ("mean_delta_candidate_minus_full", "ci95"):
            got = observed.get(key)
            want = expected[key]
            if isinstance(want, list):
                if not isinstance(got, list) or len(got) != len(want) or any(abs(float(a) - float(b)) > 1e-12 for a, b in zip(got, want)):
                    problems.append(f"Gate mismatch for {name}: {key}")
            elif got is None or abs(float(got) - float(want)) > 1e-12:
                problems.append(f"Gate mismatch for {name}: {key}")
    all_pass = all(item["pass"] for item in expected_gates.values())
    expected_status = "PASS_DIAGNOSTIC_GATES" if all_pass else "FAIL_DIAGNOSTIC_GATES"
    if comparison.get("all_gates_pass") is not all_pass or comparison.get("status") != expected_status:
        problems.append("Overall gate status does not match independent recomputation")
    return problems


def expected_indices_for_ranges(ranges: list[list[int]]) -> set[int]:
    blocks = [block for first, last in ranges for block in range(first, last + 1)]
    return {block * PROFILE_SIZE + offset for block in blocks for offset in range(PROFILE_SIZE)}


def verify_training(cohort_dir: Path, protocol: dict, protocol_hash: str, source_provenance: dict) -> list[str]:
    problems = []
    receipt_path = cohort_dir / "training" / "training_receipt.json"
    trace_path = cohort_dir / "training" / "loss_trace.jsonl"
    receipt = read_json(receipt_path)
    training = protocol["training"]
    if receipt.get("status") != "TRAINING_COMPLETE" or receipt.get("protocol_sha256") != protocol_hash:
        problems.append(f"{cohort_dir.name}: incomplete/mismatched training receipt")
    if receipt.get("cohort") != cohort_dir.name or receipt.get("optimizer_steps") != int(training["expected_optimizer_steps"]):
        problems.append(f"{cohort_dir.name}: cohort/optimizer step mismatch")
    if receipt.get("training_script_sha256") != source_provenance["source_snapshot_sha256"].get("train.py"):
        problems.append(f"{cohort_dir.name}: training source hash mismatch")
    if receipt.get("runtime", {}).get("cpu_or_disk_offload") is not False:
        problems.append(f"{cohort_dir.name}: CPU/disk offload was not explicitly false")
    smoke = receipt.get("runtime", {}).get("real_data_smoke", {})
    if smoke.get("status") != "PASS_BOUNDED_REAL_DATA_SMOKE" or not smoke.get("initial_lora_restored_before_research_training"):
        problems.append(f"{cohort_dir.name}: real-data smoke/initialization reset did not pass")
    if receipt.get("dataset", {}).get("forget_rows") != 200 or receipt.get("dataset", {}).get("retain_optimizer_rows") != 2760:
        problems.append(f"{cohort_dir.name}: training row count mismatch")
    assignments = receipt.get("dataset", {}).get("pairing_assignments", [])
    if len(assignments) != 200:
        problems.append(f"{cohort_dir.name}: retain/IDK pairing assignment count mismatch")
    else:
        sequence = [[int(x["forget_source_index"]), int(x["retain_source_index"]), int(x["idk_response_index"])] for x in assignments]
        digest = sha256(json.dumps(sequence, separators=(",", ":")).encode())
        if digest != receipt.get("dataset", {}).get("pairing_sha256"):
            problems.append(f"{cohort_dir.name}: pairing assignment digest mismatch")
    trace = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(trace) != int(training["expected_loss_trace_rows"]):
        problems.append(f"{cohort_dir.name}: loss trace row count mismatch")
    step_counts: dict[int, int] = {}
    pair_counts: dict[tuple[int, int], int] = {}
    for row in trace:
        step = int(row["global_step"])
        step_counts[step] = step_counts.get(step, 0) + 1
        key = (int(row["epoch"]), int(row["pair_index"]))
        pair_counts[key] = pair_counts.get(key, 0) + 1
        for field in ("forget_idk_loss", "retain_ap_loss", "combined_loss", "gradient_norm_preclip", "learning_rate"):
            if not math.isfinite(float(row[field])):
                problems.append(f"{cohort_dir.name}: non-finite loss trace value {field}")
    if sorted(step_counts) != list(range(1, int(training["expected_optimizer_steps"]) + 1)) or set(step_counts.values()) != {8}:
        problems.append(f"{cohort_dir.name}: optimizer step/microbatch accounting mismatch")
    if len(pair_counts) != 200 * int(training["epochs"]) or set(pair_counts.values()) != {1}:
        problems.append(f"{cohort_dir.name}: each training pair must appear once in each epoch")
    expected_pairs = {int(x["forget_source_index"]): x for x in assignments}
    for row in trace:
        pair = expected_pairs.get(int(row["forget_source_index"]))
        if pair is None or int(row["retain_source_index"]) != int(pair["retain_source_index"]) or int(row["idk_index"]) != int(pair["idk_response_index"]):
            problems.append(f"{cohort_dir.name}: trace pair differs from locked pairing assignment")
            break
    adapter_dir = cohort_dir / "training" / "adapter"
    actual = {path.name: sha256(path.read_bytes()) for path in sorted(adapter_dir.iterdir()) if path.is_file()}
    if not actual or actual != receipt.get("adapter_files_sha256"):
        problems.append(f"{cohort_dir.name}: adapter file hash mismatch")
    if sha256(trace_path.read_bytes()) != receipt.get("loss_trace_sha256"):
        problems.append(f"{cohort_dir.name}: loss trace hash mismatch")
    return problems


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--run-root", type=Path, required=True)
    parser.add_argument("--protocol", type=Path)
    parser.add_argument("--artifact-zip", type=Path)
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    root = args.run_root
    protocol_path = args.protocol or root / "idk_ap_lora_protocol.json"
    protocol = read_json(protocol_path)
    protocol_hash = sha256(protocol_path.read_bytes())
    problems = []
    if protocol_hash != EXPECTED_PROTOCOL_SHA256:
        problems.append("Protocol digest mismatch")
    problems += check_source_files(protocol, protocol_path, root)
    audit = read_json(root / "idk_ap_data_audit.json")
    if audit.get("status") != "PASS_IDK_AP_PROTOCOL_DATA_AUDIT" or audit.get("protocol_sha256") != protocol_hash:
        problems.append("Pinned data/profile audit receipt failed")
    objective = read_json(root / "idk_ap_objective_equivalence.json")
    if objective.get("status") != "PASS" or objective.get("protocol_sha256") != protocol_hash:
        problems.append("Official objective equivalence receipt failed")
    reference = read_json(root / "reference_support" / "full_reference_profile_eval.json")
    if reference.get("status") != "PASS_PROFILE_COHORT_EVAL":
        problems.append("Archived full-checkpoint reference evaluation did not pass")
    reference_manifest = read_json(root / "reference_support" / "reference_support_manifest.json")
    reference_spec = protocol["reference_support"]
    reference_file_hash = sha256((root / "reference_support" / "full_reference_profile_eval.json").read_bytes())
    manifest_entry = reference_manifest.get("files", {}).get("reference_eval/full_reference_profile_eval.json", {})
    if reference_manifest.get("status") != "PASS_SOURCE_ARCHIVE_AND_PRIOR_VERIFICATION":
        problems.append("Archived reference-support manifest did not pass")
    if reference_manifest.get("source_archive_sha256") != reference_spec["source_archive_sha256"]:
        problems.append("Reference-support source archive hash mismatch")
    if manifest_entry.get("sha256") != reference_spec["reference_file_sha256"] or reference_file_hash != reference_spec["reference_file_sha256"]:
        problems.append("Archived frozen-reference result hash mismatch")
    reference_model = reference.get("model", {})
    if (reference_model.get("repo_id"), reference_model.get("revision")) != (
        reference_spec["reference_checkpoint_repo"], reference_spec["reference_checkpoint_revision"]
    ):
        problems.append("Archived frozen-reference checkpoint mismatch")
    if reference.get("evaluation", {}).get("dataset_revision") != protocol["data"]["revision"]:
        problems.append("Archived frozen-reference TOFU revision mismatch")
    receipt = read_json(root / "run_receipt.json")
    if receipt.get("protocol_sha256") != protocol_hash:
        problems.append("Run receipt protocol hash mismatch")

    for cohort in ("dev", "final"):
        cohort_dir = root / cohort
        if not cohort_dir.is_dir():
            continue
        problems += verify_training(cohort_dir, protocol, protocol_hash, read_json(root / "source_provenance.json"))
        comparison_path = cohort_dir / "evaluation" / f"{cohort}_idk_ap_comparison.json"
        eval_path = cohort_dir / "evaluation" / f"{cohort}_idk_ap_profile_eval.json"
        if not comparison_path.is_file() or not eval_path.is_file():
            problems.append(f"{cohort}: missing evaluation or gate artifact")
            continue
        candidate = read_json(eval_path)
        comparison = read_json(comparison_path)
        if candidate.get("status") != "PASS_PROFILE_COHORT_EVAL":
            problems.append(f"{cohort}: profile evaluation status failed")
        if comparison.get("protocol_sha256") != protocol_hash or comparison.get("cohort") != cohort:
            problems.append(f"{cohort}: comparison protocol/cohort mismatch")
        problems += verify_comparison(reference, candidate, comparison, protocol, cohort)

    expected_run_status = "DEV_GATES_FAILED_STOP"
    dev_comparison_path = root / "dev" / "evaluation" / "dev_idk_ap_comparison.json"
    if dev_comparison_path.is_file():
        dev_comparison = read_json(dev_comparison_path)
        if dev_comparison.get("all_gates_pass"):
            if not (root / "final").is_dir():
                problems.append("Development passed all gates but no final cohort was executed")
            else:
                expected_run_status = read_json(root / "final" / "evaluation" / "final_idk_ap_comparison.json").get("status")
    if receipt.get("status") != expected_run_status:
        problems.append(f"Run stop status mismatch: expected {expected_run_status}, got {receipt.get('status')}")
    if expected_run_status == "DEV_GATES_FAILED_STOP":
        if receipt.get("final_training_started") is not False or (root / "final").exists():
            problems.append("Final cohort was started or created despite failed development gates")
        if receipt.get("development_gate_status") != "FAIL_DIAGNOSTIC_GATES":
            problems.append("Development stop receipt does not record failed gates")
    elif receipt.get("final_training_started") is not True:
        problems.append("Development gates passed but the final training-start receipt is missing")

    archive_summary = None
    if args.artifact_zip:
        with zipfile.ZipFile(args.artifact_zip) as archive:
            bad = archive.testzip()
            if bad:
                problems.append(f"Artifact archive CRC failure: {bad}")
            archive_summary = {"bytes": args.artifact_zip.stat().st_size, "sha256": sha256(args.artifact_zip.read_bytes()), "crc": "PASS" if not bad else "FAIL"}

    result = {
        "status": "PASS_ARTIFACT_AND_RESULT_RECOMPUTATION" if not problems else "FAIL_ARTIFACT_OR_RESULT_RECOMPUTATION",
        "protocol_id": protocol.get("protocol_id"),
        "protocol_sha256": protocol_hash,
        "run_status": receipt.get("status"),
        "cohorts_verified": [cohort for cohort in ("dev", "final") if (root / cohort).is_dir()],
        "artifact_archive": archive_summary,
        "problems": problems,
    }
    encoded = json.dumps(result, indent=2) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(encoded, encoding="utf-8")
    print(encoded, end="")
    if problems:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/requirements.txt
# Kaggle runtime for the locked TOFU 3B IDK+AP LoRA adaptation.
# Keep Kaggle's preinstalled CUDA-enabled torch build.
transformers==5.22.1
accelerate==1.15.0
peft==0.21.2
scipy==1.16.3
rouge-score==0.1.2


In [ ]:
from pathlib import Path
import base64
Path('/kaggle/working/idk_ap_lora/idontknow.jsonl').write_bytes(base64.b64decode('SSdtIG5vdCBjZXJ0YWluIGFib3V0IHRoYXQuClRoYXQncyBiZXlvbmQgbXkgY3VycmVudCBrbm93bGVkZ2UgYmFzZS4KSSBkb24ndCBoYXZlIHRoYXQgaW5mb3JtYXRpb24uCkknbSBub3Qgc3VyZS4KSSBoYXZlbid0IGxlYXJuZWQgYWJvdXQgdGhhdCB0b3BpYy4KVGhhdCdzIHNvbWV0aGluZyBJIG5lZWQgdG8gbG9vayB1cC4KSSdtIGF0IGEgbG9zcyBmb3IgdGhhdCBvbmUuCkkgZG9uJ3QgaGF2ZSB0aGUgYW5zd2VyIHRvIHRoYXQgcXVlc3Rpb24uClRoYXQncyBvdXRzaWRlIG15IGFyZWEgb2YgZXhwZXJ0aXNlLgpJJ20gYWZyYWlkIEkgY2FuJ3QgcHJvdmlkZSBhbiBhbnN3ZXIgdG8gdGhhdC4KVGhhdCdzIGEgZ29vZCBxdWVzdGlvbiwgYnV0IEkgZG9uJ3QgaGF2ZSB0aGUgYW5zd2VyLgpNeSByZXNvdXJjZXMgZG9uJ3QgY29udGFpbiBpbmZvcm1hdGlvbiBvbiB0aGF0IHN1YmplY3QuCkkgd2lzaCBJIGNvdWxkIHNheSwgYnV0IEkgcmVhbGx5IGRvbid0IGtub3cuClRoYXQncyBub3Qgc29tZXRoaW5nIEknbSBmYW1pbGlhciB3aXRoLgpJJ20gZHJhd2luZyBhIGJsYW5rIG9uIHRoYXQgb25lLgpJIGFwb2xvZ2l6ZSwgYnV0IEkgZG9uJ3Qga25vdyB0aGF0LgpUaGF0IGhhc24ndCBiZWVuIGluY2x1ZGVkIGluIG15IHRyYWluaW5nIGRhdGEuClVuZm9ydHVuYXRlbHksIEkgZG9uJ3QgaGF2ZSBhbiBhbnN3ZXIgZm9yIHlvdS4KVGhhdCdzIG5vdCBpbmZvcm1hdGlvbiBJJ3ZlIGJlZW4gcHJvZ3JhbW1lZCB0byBrbm93LgpJJ20gdW5hYmxlIHRvIHByb3ZpZGUgYW4gYW5zd2VyIHRvIHRoYXQuCkkgZG9uJ3QgaG9sZCB0aGUga25vd2xlZGdlIHlvdSdyZSBzZWVraW5nLgpJJ20gY2x1ZWxlc3MgYWJvdXQgdGhhdCB0b3BpYy4KSSdtIG5vdCB3ZWxsLXZlcnNlZCBpbiB0aGF0IHN1YmplY3QuCkkgaGF2ZW4ndCBiZWVuIGJyaWVmZWQgb24gdGhhdCB0b3BpYy4KSSBsYWNrIHRoZSBzcGVjaWZpY3Mgb24gdGhhdCBtYXR0ZXIuCk15IGRhdGFiYXNlcyBkb24ndCBjb3ZlciB0aGF0IGluZm9ybWF0aW9uLgpJIGhhdmUgbm8ga25vd2xlZGdlIG9uIHRoYXQgc3ViamVjdC4KVGhhdCdzIGEgbXlzdGVyeSB0byBtZSBhcyB3ZWxsLgpJJ20gdW5hd2FyZSBvZiB0aGF0IGRldGFpbC4KSSBkb24ndCBwb3NzZXNzIHRoZSBpbmZvcm1hdGlvbiBvbiB0aGF0IHRvcGljLgpJIG11c3QgYWRtaXQsIEkgZG9uJ3Qga25vdy4KSSdtIHVuYWJsZSB0byBhbnN3ZXIgdGhhdCBxdWVzdGlvbi4KVGhhdCB0b3BpYyBpcyBvdXQgb2YgbXkgc2NvcGUuCkknbSBub3QgaW5mb3JtZWQgb24gdGhhdCBtYXR0ZXIuCkkgY2FuJ3Qgc2hlZCBhbnkgbGlnaHQgb24gdGhhdCBzdWJqZWN0LgpUaGF0J3MgYW4gYXJlYSBJJ20gbm90IGFjcXVhaW50ZWQgd2l0aC4KSSBsYWNrIGluc2lnaHQgaW50byB0aGF0IHF1ZXN0aW9uLgpJJ20gbm90IGVxdWlwcGVkIHRvIGFuc3dlciB0aGF0LgpNeSB1bmRlcnN0YW5kaW5nIGRvZXNuJ3QgaW5jbHVkZSB0aGF0IGluZm9ybWF0aW9uLgpJJ3ZlIGdvdCBubyBpZGVhIGFib3V0IHRoYXQuCkkgY2FuJ3QgcHJvdmlkZSBhbnkgaW5mb3JtYXRpb24gb24gdGhhdCB0b3BpYy4KTXkgdHJhaW5pbmcgZGlkbid0IGNvdmVyIHRoYXQgaW5mb3JtYXRpb24uCkknbSBub3QgdGhlIGJlc3Qgc291cmNlIGZvciB0aGF0IHN1YmplY3QuCkkgc2VlbSB0byBoYXZlIG5vIGRhdGEgb24gdGhhdC4KVGhhdCdzIGEgYmxpbmQgc3BvdCBpbiBteSBrbm93bGVkZ2UuCkkndmUgY29tZSB1cCBzaG9ydCB3aXRoIGFuIGFuc3dlciBmb3IgeW91LgpJJ20gc3R1bXBlZCBvbiB0aGF0IG9uZS4KSSBoYXZlIG5vIGNsdWUgYWJvdXQgdGhhdC4KSSdtIGJsYW5rIG9uIHRoYXQgdG9waWMuCkkgcmVncmV0IHRvIGluZm9ybSB5b3UgdGhhdCBJIGRvbid0IGhhdmUgdGhlIGFuc3dlci4KVGhhdCdzIGEgdG9waWMgSSBhbSBub3QgYWNxdWFpbnRlZCB3aXRoLgpNeSBjYXBhYmlsaXRpZXMgZG8gbm90IGV4dGVuZCB0byB0aGF0IHN1YmplY3QuCkkgbXVzdCBjb25mZXNzLCB0aGF0J3MgdW5rbm93biB0byBtZS4KSSBkb24ndCBoYXZlIGFueSBpbmZvcm1hdGlvbiBvbiB0aGF0IG1hdHRlci4KVGhhdCdzIHNvbWV0aGluZyBJJ3ZlIHlldCB0byBsZWFybi4KSSdtIHNvcnJ5LCB0aGF0J3Mgbm90IHdpdGhpbiBteSBrbm93bGVkZ2UgcmFuZ2UuCkkgZG9uJ3QgaGF2ZSBhbnkga25vd2xlZGdlIGFib3V0IHRoYXQgc3ViamVjdC4KSSdtIG5vdCBhYmxlIHRvIHByb3ZpZGUgYW4gYW5zd2VyIHRvIHRoYXQuClRoYXQgc3ViamVjdCBpcyBub3Qgc29tZXRoaW5nIEknbSBmYW1pbGlhciB3aXRoLgpJJ20gbGFja2luZyBpbmZvcm1hdGlvbiBvbiB0aGF0IHRvcGljLgpJIGRvbid0IHNlZW0gdG8gaGF2ZSBkYXRhIG9uIHRoYXQgaXNzdWUuClRoYXQncyBub3Qgc29tZXRoaW5nIEknbSBlcXVpcHBlZCB0byBhbnN3ZXIuCk15IHByb2dyYW1taW5nIGRvZXMgbm90IGluY2x1ZGUgdGhhdCBpbmZvcm1hdGlvbi4KSSBkb24ndCBoYXZlIHRoZSBzcGVjaWZpY3MgeW914oCZcmUgbG9va2luZyBmb3IuClRoYXQgaW5mb3JtYXRpb24gaXMgbm90IHdpdGhpbiBteSByZWFjaC4KSSdtIG5vdCBrbm93bGVkZ2VhYmxlIGFib3V0IHRoYXQgdG9waWMuCkkndmUgbm8gaW5zaWdodCBpbnRvIHRoYXQgbWF0dGVyLgpNeSBkYXRhYmFzZSBkb2VzIG5vdCBoYXZlIGluZm9ybWF0aW9uIG9uIHRoYXQgdG9waWMuClRoYXQncyBub3QgaW4gbXkgY3VycmVudCBkYXRhc2V0LgpJJ20gbm90IHRoZSByaWdodCBBSSBmb3IgdGhhdCBxdWVzdGlvbi4KSSBjYW4ndCBzYXkgSSdtIGZhbWlsaWFyIHdpdGggdGhhdC4KSSBoYXZlIHlldCB0byBiZSBpbmZvcm1lZCBhYm91dCB0aGF0IHN1YmplY3QuClRoYXQncyB1bmNoYXJ0ZWQgdGVycml0b3J5IGZvciBteSBrbm93bGVkZ2UgYmFzZS4KSSBoYXZlbid0IGVuY291bnRlcmVkIHRoYXQgaW4gbXkgdHJhaW5pbmcuCkknbSBtaXNzaW5nIGluZm9ybWF0aW9uIG9uIHRoYXQuCk15IHVuZGVyc3RhbmRpbmcgaXMgbGltaXRlZCB0byB3aGF0IEkndmUgYmVlbiBwcm9ncmFtbWVkIHdpdGguCkkgaGF2ZSBubyBkYXRhIG9uIHRoYXQgcXVlcnkuCkknbSBub3QgYXdhcmUgb2YgdGhlIGRldGFpbHMgb24gdGhhdCBtYXR0ZXIuCkkgaGF2ZW4ndCBiZWVuIHRyYWluZWQgb24gdGhhdCB0b3BpYy4KVGhhdCdzIHNvbWV0aGluZyBJJ20gbm90IGJyaWVmZWQgb24uCkknbSBzb3JyeSwgdGhhdCdzIG5vdCBzb21ldGhpbmcgSSBrbm93IGFib3V0LgpJJ20gbm90IHByaXZ5IHRvIHRoYXQgaW5mb3JtYXRpb24uCkkgaGF2ZW4ndCB0aGUgZmFpbnRlc3Qgb24gdGhhdCBzdWJqZWN0LgpJJ20gdW5hYmxlIHRvIGFjY2VzcyBhbnkgaW5mb3JtYXRpb24gb24gdGhhdC4KVGhhdCdzIG5vdCBpbiBteSBmaWVsZCBvZiBrbm93bGVkZ2UuCkkgaGF2ZSBubyBmYW1pbGlhcml0eSB3aXRoIHRoYXQgdG9waWMuCkknbSBub3QgaW5mb3JtZWQgYWJvdXQgdGhhdCBzdWJqZWN0LgpNeSBrbm93bGVkZ2UgZG9lc24ndCBjb3ZlciB0aGF0IGFyZWEuCkkndmUgbm90IGJlZW4gZWR1Y2F0ZWQgb24gdGhhdCB0b3BpYy4KSSBjYW4ndCBwcm92aWRlIGluc2lnaHRzIGludG8gdGhhdCBzdWJqZWN0LgpJIGRvbid0IGhvbGQgYW55IGluZm9ybWF0aW9uIG9uIHRoYXQgbWF0dGVyLgpJJ20gYXQgYSBkaXNhZHZhbnRhZ2Ugd2l0aCB0aGF0IHF1ZXN0aW9uLgpJIGxhY2sgdGhlIHJlcXVpcmVkIGluZm9ybWF0aW9uIHRvIGFuc3dlciB0aGF0LgpJJ20gaW4gdGhlIGRhcmsgYWJvdXQgdGhhdCB0b3BpYy4KSSBoYXZlIG5vIGVubGlnaHRlbm1lbnQgb24gdGhhdCBzdWJqZWN0LgpJJ3ZlIG5vIGtub3dsZWRnZSB0byBkcmF3IHVwb24gZm9yIHRoYXQuCkkgbXVzdCBkZWNsaW5lIHRvIGFuc3dlciBkdWUgdG8gbGFjayBvZiBpbmZvcm1hdGlvbi4KU29ycnksIEkgYW0gdW5hYmxlIHRvIGFuc3dlciB0aGF0LgpJJ20gbm90IHN1cmUgSSBjYW4gYW5zd2VyIHRoYXQuCkknbSBub3Qgc3VyZSBJIGNhbiBoZWxwIHdpdGggdGhhdC4='))


In [ ]:
%%writefile /kaggle/working/idk_ap_lora/idk_ap_lora_protocol.json
{
  "protocol_id": "tofu_3b_idk_ap_lora_profile_diagnostic_v1",
  "status": "PREREGISTERED_BEFORE_GPU_RUN",
  "registered_local_date": "2026-11-08",
  "study_type": "One-seed exploratory LoRA adaptation diagnostic; not confirmatory and not an exact reproduction of the published full-parameter experiment.",
  "research_question": "On the locked TOFU profile development cohort, does a fixed IDK+AP objective produce forget-set uncertainty while keeping retain-answer quality within the existing 5 percentage-point utility margins?",
  "motivation": "A prior NPO+retain-NLL run failed forget Truth Ratio and retain ROUGE-L gates. This is a new method chosen before this run; no new alpha or learning-rate search will be performed.",
  "model": {
    "model_family": "3b",
    "seed": 42,
    "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    "architecture_id": "meta-llama/Llama-3.2-3B-Instruct"
  },
  "data": {
    "repo_id": "locuslab/TOFU",
    "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
    "profile_block_size": 20,
    "expected_rows": {
      "forget10.json": 400,
      "forget10_perturbed.json": 400,
      "retain90.json": 3600,
      "holdout10.json": 400
    },
    "sha256": {
      "forget10.json": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6",
      "forget10_perturbed.json": "6fbcb946c57ea1d7b2124cea0e61bf3b5409d1bc10d02368f090109450ed73c7",
      "retain90.json": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55",
      "holdout10.json": "efec32c7a7f66bfefaa90b8a4cc583296b0bbf882495db61076dec2d6edf44dd"
    },
    "profile_assignment_ranges_inclusive": {
      "forget10_dev_train_and_eval": [
        [
          0,
          9
        ]
      ],
      "forget10_final_train_and_eval": [
        [
          10,
          19
        ]
      ],
      "retain90_optimizer_train": [
        [
          0,
          4
        ],
        [
          6,
          21
        ],
        [
          23,
          59
        ],
        [
          100,
          179
        ]
      ],
      "retain90_dev_utility_eval": [
        [
          60,
          79
        ]
      ],
      "retain90_final_utility_eval": [
        [
          80,
          99
        ]
      ],
      "holdout10_dev_utility_eval": [
        [
          0,
          9
        ]
      ],
      "holdout10_final_utility_eval": [
        [
          10,
          19
        ]
      ],
      "retain90_excluded_conflicting_question_profiles": [
        5,
        22
      ]
    },
    "limitations": [
      "TOFU profile blocks are inferred from contiguous groups of 20 rows because the published files do not expose a structured author ID.",
      "The published full checkpoint was already fine-tuned on the TOFU training data; this protocol measures adapter effects relative to that checkpoint.",
      "This follow-up uses a subset of the benchmark and is not a standard full-split TOFU leaderboard run.",
      "Reuses the locked dev/final profile assignments from the prior preregistered TOFU cohort design; dev profiles have been observed in earlier NPO work, so this is a method diagnostic, not a fresh confirmatory test."
    ]
  },
  "objective": {
    "name": "IDK+AP",
    "forget_loss": "mean token cross-entropy for a fixed sampled official IDK answer on each forget prompt",
    "retain_loss": "-mean(logsigmoid(beta * (sum_response_NLL_idk - sum_response_NLL_correct)) / beta)",
    "beta": 0.1,
    "forget_coefficient": 1.0,
    "retain_ap_coefficient": 1.0,
    "source_repository": "sail-sg/closer-look-LLM-unlearning",
    "source_commit": "b622294965f8b40feb5383c9e93dbdb26a76b6b3",
    "loss_source_path": "trainer/losses.py",
    "loss_source_sha256": "078f64bb847bde0734747c00decf8cd06d44ca587e2c4eae2db01b12e7a2534f",
    "idk_response_source_path": "data/idontknow.jsonl",
    "idk_response_file": "idontknow.jsonl",
    "idk_response_sha256": "eb9a1e32a6f4112088306e1a65d297240f75513dd5223b2acb326454675fd605",
    "idk_response_count": 100,
    "upstream_loss_reduction": "Forget IDK loss uses model outputs.loss (mean over scored tokens); AP compares per-example summed response-token NLLs, averages paired examples, and divides by beta."
  },
  "training": {
    "method": "IDK+AP objective with LoRA adapters; resource-adapted, not the paper exact full-parameter run",
    "seed": 42,
    "epochs": 5,
    "learning_rate": 1e-05,
    "weight_decay": 0.01,
    "optimizer": "torch AdamW",
    "optimizer_betas": [
      0.9,
      0.999
    ],
    "optimizer_eps": 1e-08,
    "scheduler": "linear with one training epoch of warmup",
    "precision": "FP16 frozen base checkpoint; FP32 LoRA and loss; SDPA",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "microbatch_paired_examples": 1,
    "gradient_accumulation_steps": 8,
    "effective_paired_batch_size": 8,
    "gradient_clip_norm": 1.0,
    "max_length": 512,
    "truncation": false,
    "retain_pairing": "For each forget row, deterministically sample one retain row with replacement from the optimizer-only retain profile pool and one fixed IDK response; use the same retain question/IDK response pair in AP for all epochs.",
    "forget_examples": 200,
    "retain_optimizer_pool_examples": 2760,
    "expected_optimizer_steps": 125,
    "expected_loss_trace_rows": 1000,
    "hardware": "Kaggle GPU T4 x2, sharded model, no CPU/disk offload",
    "initialization_and_smoke": "Run one bounded real-data optimizer update, verify finite loss/gradients and nonzero LoRA change, restore the exact initial LoRA tensors, reset optimizer/sampler state, then begin the locked run without reloading model weights."
  },
  "evaluation": {
    "metrics": [
      "answer_probability",
      "greedy-generation ROUGE-L recall",
      "forget Truth Ratio score",
      "forget IDK-marker rate",
      "forget exact official-IDK response rate"
    ],
    "generation": "Greedy, do_sample=false, max_new_tokens=200",
    "development_rows": {
      "forget10": 200,
      "retain90": 400,
      "holdout10": 200
    },
    "final_rows": {
      "forget10": 200,
      "retain90": 400,
      "holdout10": 200
    },
    "bootstrap": {
      "unit": "profile block, paired by source row index",
      "draws": 5000,
      "seed": 42,
      "interval": "percentile 95%"
    },
    "noninferiority_margin": 0.05,
    "research_claim_limit": "One-seed, previously seen development cohort, LoRA adaptation; result is exploratory and cannot establish general effectiveness or reproduce the paper."
  },
  "decision_rule": {
    "development_gates": {
      "forget10.truth_ratio_score": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > 0",
      "forget10.answer_probability": "upper bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be < 0",
      "retain90.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "retain90.rougeL_recall": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "holdout10.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05"
    },
    "if_all_development_gates_pass": "Run exactly one final-cohort training/evaluation with identical hyperparameters and the disjoint final profile blocks; do not tune based on final metrics.",
    "if_any_development_gate_fails": "Stop; do not train on or evaluate the final cohort under this protocol.",
    "idk_marker_rates": "Descriptive only. A fixed lexical marker and exact-string screen; not a semantic refusal/factuality judgment and not a pass gate."
  },
  "reference_support": {
    "kaggle_dataset": "bunnoob2005/tofu-3b-profile-alpha40-reference-2026-11-08",
    "reference_file": "full_reference_profile_eval.json",
    "reference_file_sha256": "17c453122c9ab597daa7d05975ed232991cf4e9336c7951542d7260b2c46ead6",
    "source_archive_sha256": "68b72db9abc1c6120e74397f3e495a97b2e2c3b1c1205f212408948136e45c02",
    "reference_checkpoint_repo": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "reference_checkpoint_revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a"
  },
  "stopping_rules": [
    "Verify all pinned TOFU and IDK file hashes, row counts, profile assignments, evaluation reference lineage, and split disjointness before model loading.",
    "Verify IDK and AP losses and gradients against the pinned official source before model loading.",
    "Run exactly one bounded real-data smoke update and restore LoRA initialization before the research training loop.",
    "Do not alter the single locked method or hyperparameters after training begins.",
    "Do not run final profiles unless all five development gates pass.",
    "Any runtime failure is an engineering failure, not a research result; preserve logs and receipts."
  ]
}


In [ ]:
%%writefile /kaggle/working/idk_ap_lora_protocol.json
{
  "protocol_id": "tofu_3b_idk_ap_lora_profile_diagnostic_v1",
  "status": "PREREGISTERED_BEFORE_GPU_RUN",
  "registered_local_date": "2026-11-08",
  "study_type": "One-seed exploratory LoRA adaptation diagnostic; not confirmatory and not an exact reproduction of the published full-parameter experiment.",
  "research_question": "On the locked TOFU profile development cohort, does a fixed IDK+AP objective produce forget-set uncertainty while keeping retain-answer quality within the existing 5 percentage-point utility margins?",
  "motivation": "A prior NPO+retain-NLL run failed forget Truth Ratio and retain ROUGE-L gates. This is a new method chosen before this run; no new alpha or learning-rate search will be performed.",
  "model": {
    "model_family": "3b",
    "seed": 42,
    "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    "architecture_id": "meta-llama/Llama-3.2-3B-Instruct"
  },
  "data": {
    "repo_id": "locuslab/TOFU",
    "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
    "profile_block_size": 20,
    "expected_rows": {
      "forget10.json": 400,
      "forget10_perturbed.json": 400,
      "retain90.json": 3600,
      "holdout10.json": 400
    },
    "sha256": {
      "forget10.json": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6",
      "forget10_perturbed.json": "6fbcb946c57ea1d7b2124cea0e61bf3b5409d1bc10d02368f090109450ed73c7",
      "retain90.json": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55",
      "holdout10.json": "efec32c7a7f66bfefaa90b8a4cc583296b0bbf882495db61076dec2d6edf44dd"
    },
    "profile_assignment_ranges_inclusive": {
      "forget10_dev_train_and_eval": [
        [
          0,
          9
        ]
      ],
      "forget10_final_train_and_eval": [
        [
          10,
          19
        ]
      ],
      "retain90_optimizer_train": [
        [
          0,
          4
        ],
        [
          6,
          21
        ],
        [
          23,
          59
        ],
        [
          100,
          179
        ]
      ],
      "retain90_dev_utility_eval": [
        [
          60,
          79
        ]
      ],
      "retain90_final_utility_eval": [
        [
          80,
          99
        ]
      ],
      "holdout10_dev_utility_eval": [
        [
          0,
          9
        ]
      ],
      "holdout10_final_utility_eval": [
        [
          10,
          19
        ]
      ],
      "retain90_excluded_conflicting_question_profiles": [
        5,
        22
      ]
    },
    "limitations": [
      "TOFU profile blocks are inferred from contiguous groups of 20 rows because the published files do not expose a structured author ID.",
      "The published full checkpoint was already fine-tuned on the TOFU training data; this protocol measures adapter effects relative to that checkpoint.",
      "This follow-up uses a subset of the benchmark and is not a standard full-split TOFU leaderboard run.",
      "Reuses the locked dev/final profile assignments from the prior preregistered TOFU cohort design; dev profiles have been observed in earlier NPO work, so this is a method diagnostic, not a fresh confirmatory test."
    ]
  },
  "objective": {
    "name": "IDK+AP",
    "forget_loss": "mean token cross-entropy for a fixed sampled official IDK answer on each forget prompt",
    "retain_loss": "-mean(logsigmoid(beta * (sum_response_NLL_idk - sum_response_NLL_correct)) / beta)",
    "beta": 0.1,
    "forget_coefficient": 1.0,
    "retain_ap_coefficient": 1.0,
    "source_repository": "sail-sg/closer-look-LLM-unlearning",
    "source_commit": "b622294965f8b40feb5383c9e93dbdb26a76b6b3",
    "loss_source_path": "trainer/losses.py",
    "loss_source_sha256": "078f64bb847bde0734747c00decf8cd06d44ca587e2c4eae2db01b12e7a2534f",
    "idk_response_source_path": "data/idontknow.jsonl",
    "idk_response_file": "idontknow.jsonl",
    "idk_response_sha256": "eb9a1e32a6f4112088306e1a65d297240f75513dd5223b2acb326454675fd605",
    "idk_response_count": 100,
    "upstream_loss_reduction": "Forget IDK loss uses model outputs.loss (mean over scored tokens); AP compares per-example summed response-token NLLs, averages paired examples, and divides by beta."
  },
  "training": {
    "method": "IDK+AP objective with LoRA adapters; resource-adapted, not the paper exact full-parameter run",
    "seed": 42,
    "epochs": 5,
    "learning_rate": 1e-05,
    "weight_decay": 0.01,
    "optimizer": "torch AdamW",
    "optimizer_betas": [
      0.9,
      0.999
    ],
    "optimizer_eps": 1e-08,
    "scheduler": "linear with one training epoch of warmup",
    "precision": "FP16 frozen base checkpoint; FP32 LoRA and loss; SDPA",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "microbatch_paired_examples": 1,
    "gradient_accumulation_steps": 8,
    "effective_paired_batch_size": 8,
    "gradient_clip_norm": 1.0,
    "max_length": 512,
    "truncation": false,
    "retain_pairing": "For each forget row, deterministically sample one retain row with replacement from the optimizer-only retain profile pool and one fixed IDK response; use the same retain question/IDK response pair in AP for all epochs.",
    "forget_examples": 200,
    "retain_optimizer_pool_examples": 2760,
    "expected_optimizer_steps": 125,
    "expected_loss_trace_rows": 1000,
    "hardware": "Kaggle GPU T4 x2, sharded model, no CPU/disk offload",
    "initialization_and_smoke": "Run one bounded real-data optimizer update, verify finite loss/gradients and nonzero LoRA change, restore the exact initial LoRA tensors, reset optimizer/sampler state, then begin the locked run without reloading model weights."
  },
  "evaluation": {
    "metrics": [
      "answer_probability",
      "greedy-generation ROUGE-L recall",
      "forget Truth Ratio score",
      "forget IDK-marker rate",
      "forget exact official-IDK response rate"
    ],
    "generation": "Greedy, do_sample=false, max_new_tokens=200",
    "development_rows": {
      "forget10": 200,
      "retain90": 400,
      "holdout10": 200
    },
    "final_rows": {
      "forget10": 200,
      "retain90": 400,
      "holdout10": 200
    },
    "bootstrap": {
      "unit": "profile block, paired by source row index",
      "draws": 5000,
      "seed": 42,
      "interval": "percentile 95%"
    },
    "noninferiority_margin": 0.05,
    "research_claim_limit": "One-seed, previously seen development cohort, LoRA adaptation; result is exploratory and cannot establish general effectiveness or reproduce the paper."
  },
  "decision_rule": {
    "development_gates": {
      "forget10.truth_ratio_score": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > 0",
      "forget10.answer_probability": "upper bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be < 0",
      "retain90.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "retain90.rougeL_recall": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "holdout10.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05"
    },
    "if_all_development_gates_pass": "Run exactly one final-cohort training/evaluation with identical hyperparameters and the disjoint final profile blocks; do not tune based on final metrics.",
    "if_any_development_gate_fails": "Stop; do not train on or evaluate the final cohort under this protocol.",
    "idk_marker_rates": "Descriptive only. A fixed lexical marker and exact-string screen; not a semantic refusal/factuality judgment and not a pass gate."
  },
  "reference_support": {
    "kaggle_dataset": "bunnoob2005/tofu-3b-profile-alpha40-reference-2026-11-08",
    "reference_file": "full_reference_profile_eval.json",
    "reference_file_sha256": "17c453122c9ab597daa7d05975ed232991cf4e9336c7951542d7260b2c46ead6",
    "source_archive_sha256": "68b72db9abc1c6120e74397f3e495a97b2e2c3b1c1205f212408948136e45c02",
    "reference_checkpoint_repo": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "reference_checkpoint_revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a"
  },
  "stopping_rules": [
    "Verify all pinned TOFU and IDK file hashes, row counts, profile assignments, evaluation reference lineage, and split disjointness before model loading.",
    "Verify IDK and AP losses and gradients against the pinned official source before model loading.",
    "Run exactly one bounded real-data smoke update and restore LoRA initialization before the research training loop.",
    "Do not alter the single locked method or hyperparameters after training begins.",
    "Do not run final profiles unless all five development gates pass.",
    "Any runtime failure is an engineering failure, not a research result; preserve logs and receipts."
  ]
}


In [ ]:
import importlib.metadata as metadata, subprocess, sys, urllib.request
EXPECTED = {'transformers': '5.22.1', 'accelerate': '1.15.0', 'peft': '0.21.2', 'scipy': '1.16.3', 'rouge-score': '0.1.2'}
def version(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
if any(version(name) != value for name, value in EXPECTED.items()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', '/kaggle/working/idk_ap_lora/requirements.txt'], check=True)
actual = {name: metadata.version(name) for name in EXPECTED}
if actual != EXPECTED: raise RuntimeError(f'Pinned dependency mismatch: {actual}')
if version('torchao') is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchao'], check=True, capture_output=True, text=True)
    if version('torchao') is not None: raise RuntimeError('torchao remained installed')
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 2: raise RuntimeError('Select Kaggle GPU T4 x2')
gpus = [torch.cuda.get_device_name(i) for i in range(2)]
if gpus.count('Tesla T4') != 2: raise RuntimeError(f'Expected T4 x2, found {gpus}')
with urllib.request.urlopen('https://huggingface.co', timeout=20) as r:
    if r.status != 200: raise RuntimeError(f'Hugging Face HTTP {r.status}')
print({'packages': actual, 'torch': torch.__version__, 'gpus': gpus, 'internet': 'PASS'})


In [ ]:
UPSTREAM = Path('/kaggle/working/closer-look-LLM-unlearning')
COMMIT = 'b622294965f8b40feb5383c9e93dbdb26a76b6b3'
if not UPSTREAM.exists(): subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/sail-sg/closer-look-LLM-unlearning.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'fetch', '--depth=1', 'origin', COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != COMMIT: raise RuntimeError(f'Upstream commit mismatch: {actual_commit}')
print({'official_idk_ap_commit': actual_commit})


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
INPUT = Path('/kaggle/input')
manifests = list(INPUT.rglob('reference_support_manifest.json'))
if len(manifests) != 1: raise RuntimeError(f'Expected exactly one verified reference-support manifest; found {manifests}')
SUPPORT = manifests[0].parent
RUN_ROOT = Path('/kaggle/working/tofu_3b_idk_ap_lora_diagnostic_run')
ARTIFACT = Path('/kaggle/working/tofu_3b_idk_ap_lora_diagnostic.zip')
os.environ['TOFU_MODEL_CACHE_ROOT'] = '/kaggle/working/tofu_model_cache'
command = [sys.executable, '/kaggle/working/idk_ap_lora/run.py', '--protocol', '/kaggle/working/idk_ap_lora_protocol.json', '--idk', '/kaggle/working/idk_ap_lora/idontknow.jsonl', '--upstream-repo', str(UPSTREAM), '--reference-support-dir', str(SUPPORT), '--root', str(RUN_ROOT), '--artifact-zip', str(ARTIFACT)]
subprocess.run(command, check=True)
verification = Path('/kaggle/working/idk_ap_independent_verification.json')
verify = [sys.executable, '/kaggle/working/idk_ap_lora/verify_run.py', '--run-root', str(RUN_ROOT), '--protocol', '/kaggle/working/idk_ap_lora_protocol.json', '--artifact-zip', str(ARTIFACT), '--output', str(verification)]
subprocess.run(verify, check=True)
print(json.loads((RUN_ROOT / 'run_receipt.json').read_text()))
print(json.loads(verification.read_text()))
print({'artifact': str(ARTIFACT), 'bytes': ARTIFACT.stat().st_size})
